# Deep-Mechinterp Starter: AtomicDance Planner — v2 (reviewt & gefixt)

Aufbau wie gehabt: Stimulus-Suite → Aktivierungs-Hooks → Kontrastanalyse → lineare Probes → D3PM-Trajektorie → Kausal-Skelett. Das echte, selbst trainierte Planner-Modell wird **nur lesend** von Drive geladen; der Stimulus ist Musik (Faktoren aus den AIST++-Sequenznamen).

**Änderungen in v2 (nach Review):**

1. **Modell-Rekonstruktion gefixt (Abschnitt 5).** Die Auto-Discovery ist durch die verifizierte direkte Konstruktion ersetzt: `UniformD3PM(AtomicPlannerTransformer(num_atomic_classes=100, music_dim=35, max_seq_len=150))`. Im Quellcode gilt `num_classes = num_atomic_classes + 1` → Checkpoint-Embedding `[101, 512]` ⇒ **K=100**. `max_seq_len=150` ist zwingend (Klassen-Default 18000 ergäbe einen `pe`-Buffer `[18000, 1, 512]` → Size-Mismatch). Lokal gegen den Checkpoint-Abdruck bestätigt: 110 Tensoren, 19.121.965 Elemente, `strict=True` lädt.
   *Nebenbefund: `load_state_dict(strict=False)` wirft bei Size-Mismatch trotzdem `RuntimeError` — daran starb die alte K-Schleife, obwohl K=100 vermutlich schon erfolgreich geladen hatte.*
2. **Parameterdiskrepanz erklärt:** 19.121.965 (State-Dict) = 19.044.965 trainierbar + 76.800 (`pe [150, 1, 512]`) + 200 (`alphas` + `alpha_bars`). Kein Problem — Buffer vs. Parameter.
3. **Probe-Split gefixt:** `GroupShuffleSplit` gruppiert jetzt nach **Musik-ID** statt `base_seq`. Der Probe-Input ist ausschließlich Musik, und derselbe Song wird in AIST++ von mehreren Tänzern getanzt — sonst landet identischer Input in Train **und** Test (Leakage).
4. **Situations-Probe = Negativkontrolle:** sBM/sFM teilen dieselben Songs; erwartet ist Chance-Niveau. Deutlich mehr ⇒ Leakage-/Artefakt-Alarm.
5. **Trajektorie über mehrere Sequenzen gemittelt**; Chance- **und** Majority-Baseline im Probe-Plot; Baseline-Zuordnung über Facet-Titel (vorher konnte die Chance-Linie im falschen Facet landen).

**v2.1 (nach dem ersten Lauf):** Der Test-Split enthält **nur sBM-Fenster** (der Lauf zeigte `situation: 317/0`) — `run_probe` überspringt jetzt Labels mit < 2 Klassen (daran crashte Abschnitt 12), eine **Genre-Permutations-Probe** ersetzt die entfallene Situations-Negativkontrolle, und das Stimulus-Sampling kommt ohne `groupby.apply`-DeprecationWarning aus.

**v3 (Rhythmus-Zeit-Probes, Abschnitte 15–16):** Konsequenz aus dem ersten Probe-Lauf: Genre/Tempo-Accuracy 0.0 war ein **Split-Artefakt** (Musik-ID ≡ Genre im Test-Split, Details in Abschnitt 15). Neu: frameweise **Rhythmus-Probes** (Beat als Positiv-Kontrolle; Beat-Phase, lokales Tempo, Frames-bis-Beat als Integrationstargets) mit Input-, Kontext- und Positions-Baselines, **Counterfactual-Beat-Shift**, **Transition↔Beat-Alignment** als Verhaltens-Readout und **Settledness-Probe** (latente Uhr bei festem t). Alle Targets variieren innerhalb jedes Songs — der strenge Musik-ID-Split bleibt, Aliasing ist unmöglich.

**v3.1 (nach dem v3-Lauf):** Der Lauf ergab (a) *keine* linear lesbare Phase (Layer-R² ≤ 0 bei Fenster-Baseline 0.68) und (b) Settledness-AUC 0.90–0.99. Zwei Entscheidungs-Zellen dazu: **15e** Linear-vs-MLP-Gap (fehlt die Phase wirklich, oder ist sie nur nichtlinear kodiert?) und **16b** Label-Identitäts-Kontrolle (ist die latente Uhr echt, oder liest die Probe bei transition-lastigen Plänen nur „aktuelles Label = 0"? — Label-One-Hot-Baseline + Movement-only-Auswertung). Zelle 16 speichert dafür jetzt zusätzlich y_t und den finalen Plan.

**v3.2 (Abschnitt 17):** Der v3.1-Lauf bestätigte: Phase fehlt wirklich (auch MLP-Probes ≤ 0), und die Settledness-AUC steht auf der Kippe (Label-Baseline 0.937 vs. Aktivierungen 0.960 bei Schritt 10). Abschnitt 17 untersucht die Kippe mechanistisch mit dem Werkzeug aus `Erikiss/explaining_attention_heads`: symbolische Attention-Programme je Head (Soft-IoU-Bestfit) + Replacement-Experimente (Uniform-Ablation vs. Programm-Ersatz) mit dem Settledness-Gap als Damage-Metrik.

**v4 (Abschnitt 18):** Der 17b-Lauf entschied: 16b-Uhr ist **echt** (+0.23 Movement-Gap über Label-Baseline), aber auf Head-Ebene **nicht lokalisierbar** (max. −0.014 Einzel-Head-Schaden; Layer 5–7 wegen Readout-Position ungetestet). Superpositions-Verdacht → Abschnitt 18 wendet **SPD** (Stochastic Parameter Decomposition, Goodfire AI, via `Erikiss/param-decomp`/`nano_param_decomp`) auf die MLP-Matrizen der Layer 0–4 an: Rang-1-Komponenten + CI-Transformer, dann Komponenten-Ablation gegen den Settledness-Gap — konzentriert (sparsames Uhr-Subnetz) vs. flach (echte Redundanz).

**v5 (Abschnitt 19):** Der v4.1-SPD-Lauf ergab: Uhr = superponiertes Richtungs-Ensemble, konzentriert in **L4.linear2** (55 % des Schadens in Layer 4; Top-Komponente c63 allein −0.027), Top-Komponenten feuern **invertiert** (stärker auf *unsettled* Frames — Instabilitäts-Detektor). Abschnitt 19 nimmt die **Jacobian-Lens** (`Erikiss/jacobian-lens`, Global-Workspace-Referenzcode): J_l-Transport in die Endbasis via Kotangenten-Estimator, **J-Space-Spektren** (eine dominante orthogonale Richtung in den Mid-Layern?), **helle Punkte** (Layer×Frame-Agreement mit der finalen Vorhersage, Link zur Settledness) und **Geometrie-Variation** (Skalierung/Rotation-im-J-Raum/Orth-Translation) zur Trennung von invarianter Grammatik und geometrie-gekoppelter Logik.

Alle Ergebnisse (CSV/NPZ/PNG) landen dauerhaft unter `MyDrive/AtomicDance/mechinterp/`. Laufzeit: **T4 reicht völlig** (19-Mio-Parameter-Modell).

## 1) Runtime-Check

In [ ]:
import shutil, subprocess, sys, platform
print("Python:", sys.version.split()[0], "|", platform.platform())
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])
else:
    print("Keine GPU aktiv — Analyse laeuft auch auf CPU, nur langsamer.")

## 2) Setup: Drive, Ordner, Seeds, Pakete

In [ ]:
%pip install -q einops scikit-learn seaborn

In [ ]:
from google.colab import drive
from pathlib import Path
import os, random
import numpy as np
import torch

drive.mount('/content/drive')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DRIVE = Path('/content/drive/MyDrive/AtomicDance')
MI = DRIVE / 'mechinterp'
DATA_DIR, ACT_DIR, OUT_DIR = MI / 'data', MI / 'activations', MI / 'outputs'
for p in [DATA_DIR, ACT_DIR, OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch:', torch.__version__, '| device:', DEVICE)
print('Outputs →', MI)

## 3) Repo klonen & Testdaten aus dem Drive-Cache

Geklont wird **dein Fork** (`Erikiss/AtomicDance`, Stand: identisch mit `oceanflowlab/AtomicDance`, Commit `d790dbc`) — so bleibt der Lauf reproduzierbar, auch wenn sich das Original-Repo später ändert.

In [ ]:
%cd /content
!rm -rf /content/AtomicDance
!git clone -q https://github.com/Erikiss/AtomicDance.git
%cd /content/AtomicDance

In [ ]:
import glob, os, shutil, zipfile, tarfile

archives = sorted(glob.glob(str(DRIVE / 'dataset' / '*')))
assert archives, 'Kein Datensatz-Archiv im Drive-Cache (MyDrive/AtomicDance/dataset).'
src = archives[0]
dst = '/content/AtomicDance/data/atomic_aistpp'
if not os.path.exists(os.path.join(dst, 'manifest.json')):
    tmp = '/content/_atomic_extract'
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    (zipfile.ZipFile(src).extractall(tmp) if src.endswith('.zip')
     else tarfile.open(src).extractall(tmp))
    hit = next(r for r, d, f in os.walk(tmp) if 'manifest.json' in f)
    os.makedirs('/content/AtomicDance/data', exist_ok=True)
    shutil.rmtree(dst, ignore_errors=True)
    shutil.move(hit, dst)
print('Datensatz bereit:', dst)

test_music = np.load(os.path.join(dst, 'test', 'music.npy'))
test_labels = np.load(os.path.join(dst, 'test', 'labels.npy'))
import json as _json
with open(os.path.join(dst, 'test', 'names.json')) as f:
    test_names = _json.load(f)
print('music:', test_music.shape, '| labels:', test_labels.shape, '| names:', len(test_names))
print('Beispielname:', test_names[0])

## 4) Checkpoint-Inspektion (nur lesend)

Neuesten Planner-Checkpoint von Drive laden und hineinschauen: Keys, Trainings-Args, Parameterzahl. **Erwartung:** 19.121.965 State-Dict-Elemente = 19.044.965 trainierbare Parameter (Trainingslog) + 77.000 Buffer (`pe [150, 1, 512]` = 76.800, `alphas`/`alpha_bars` = 200).

In [ ]:
ckpts = sorted((DRIVE / 'runs' / 'atomic_planner').glob('*.pt'), key=os.path.getmtime)
assert ckpts, 'Kein Planner-Checkpoint auf Drive gefunden.'
CKPT_PATH = ckpts[-1]
print('Checkpoint:', CKPT_PATH)

try:
    ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
except TypeError:
    ckpt = torch.load(CKPT_PATH, map_location='cpu')

print('Top-Level-Keys:', list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt))

def find_state_dict(obj):
    if isinstance(obj, dict):
        for k in ['model', 'model_state', 'model_state_dict', 'state_dict', 'ema', 'ema_model']:
            if k in obj and isinstance(obj[k], dict):
                return obj[k], k
        if obj and all(torch.is_tensor(v) for v in obj.values()):
            return obj, '<root>'
    raise RuntimeError('State-Dict im Checkpoint nicht identifiziert — Keys oben pruefen.')

STATE, state_key = find_state_dict(ckpt)
STATE = { (k[7:] if k.startswith('module.') else k): v for k, v in STATE.items() }
n_params = sum(v.numel() for v in STATE.values() if torch.is_tensor(v))
print(f'State-Dict unter "{state_key}" | Tensoren: {len(STATE)} | Elemente: {n_params:,}')

CKPT_ARGS = {}
for k in ['args', 'config', 'hparams']:
    if isinstance(ckpt, dict) and k in ckpt:
        a = ckpt[k]
        CKPT_ARGS = dict(vars(a)) if hasattr(a, '__dict__') else dict(a)
        break
print('Trainings-Args:', CKPT_ARGS if CKPT_ARGS else '(keine im Checkpoint — nutze README-Defaults)')
print('\nErste State-Keys:', list(STATE.keys())[:8])

## 5) Planner rekonstruieren & Gewichte laden (verifiziert, v2)

Keine Discovery mehr nötig — `model/atomic_planner.py` definiert:

- `AtomicPlannerTransformer(num_atomic_classes, music_dim, latent_dim=512, num_layers=8, num_heads=8, ff_size=1024, dropout=0.1, max_seq_len=18000)` mit intern **`num_classes = num_atomic_classes + 1`**
- `UniformD3PM(model, num_steps=100, schedule='cosine')`

Die Konstruktion unten spiegelt exakt `train_atomic.py` und nutzt die im Checkpoint gespeicherten Trainings-Args (README-Defaults als Fallback). `schedule` ist egal — die Buffer werden vom Checkpoint überschrieben.

In [ ]:
import sys
import torch.nn as nn

sys.path.insert(0, '/content/AtomicDance')
from model.atomic_planner import AtomicPlannerTransformer, UniformD3PM

# Verifiziert: Checkpoint label_embedding [101,512] => num_atomic_classes=100.
# max_seq_len=150 zwingend (Default 18000 gaebe pe-Buffer [18000,1,512] -> Size-Mismatch).
PLANNER = UniformD3PM(
    AtomicPlannerTransformer(
        num_atomic_classes=int(CKPT_ARGS.get('num_classes', 100)),
        music_dim=int(CKPT_ARGS.get('music_dim', 35)),
        latent_dim=int(CKPT_ARGS.get('latent_dim', 512)),
        num_layers=int(CKPT_ARGS.get('num_layers', 8)),
        num_heads=int(CKPT_ARGS.get('num_heads', 8)),
        ff_size=int(CKPT_ARGS.get('ff_size', 1024)),
        dropout=float(CKPT_ARGS.get('dropout', 0.1)),
        max_seq_len=int(CKPT_ARGS.get('seq_len', 150)),
    ),
    num_steps=int(CKPT_ARGS.get('diffusion_steps', 100)),
)
PLANNER.load_state_dict(STATE, strict=True)
PLANNER.to(DEVICE).eval()

n_train = sum(p.numel() for p in PLANNER.parameters())
n_all = sum(v.numel() for v in PLANNER.state_dict().values())
print(f'✔ UniformD3PM(AtomicPlannerTransformer), K={PLANNER.num_classes - 1}')
print(f'  trainierbare Parameter: {n_train:,} (Soll: 19.044.965)')
print(f'  State-Dict inkl. Buffer: {n_all:,} (Soll: 19.121.965)')

## 6) Forward-Check

Die Signatur ist aus dem Repo bekannt: `forward(noisy_labels, music_features, timesteps, padding_mask=None)`. Standard-Analysezustand: `y` = uniformes Rauschen (fester Seed), `t = T_MAX − 1` — das Modell sieht dann effektiv nur die Musik, ideal für Musik-Probes.

In [ ]:
import inspect

CORE = PLANNER.model
print('Gehookter Kern:', type(CORE).__name__)
print('forward-Signatur:', inspect.signature(CORE.forward))

NUM_CLASSES = CORE.num_classes            # 101: Label 0 = Transition, 1-100 = Movements
SEQ_LEN = test_music.shape[1]             # 150
T_MAX = int(PLANNER.num_steps)            # 100

g = torch.Generator().manual_seed(SEED)
NOISE_Y = torch.randint(0, NUM_CLASSES, (1, SEQ_LEN), generator=g)

def core_forward(music_batch, y=None, t=None):
    B = music_batch.shape[0]
    y = (NOISE_Y.repeat(B, 1) if y is None else y).to(DEVICE)
    t = torch.full((B,), T_MAX - 1 if t is None else t, dtype=torch.long, device=DEVICE)
    m = music_batch.to(DEVICE).float()
    return CORE(y, m, t)  # forward(noisy_labels, music_features, timesteps, padding_mask=None)

with torch.no_grad():
    logits = core_forward(torch.tensor(test_music[:2]))
print('Logits-Shape:', tuple(logits.shape), '(erwartet: (2, 150, 101))')
assert logits.shape == (2, SEQ_LEN, NUM_CLASSES), 'Unerwartete Ausgabeform — bitte melden.'

## 7) Stimulus-Suite: Faktoren aus den Sequenznamen

Aus jedem Testfenster-Namen (`gBR_sBM_cAll_d04_mBR0_ch01_slice0`) parsen wir: **Genre** (10 Klassen), **Situation** (Basic/Advanced), **Tänzer**, **Musikstück** und die **Tempoklasse** (letzte Ziffer der Musik-ID; in der AIST-DB entspricht 0–5 aufsteigendem BPM von ca. 80–130).

In [ ]:
import pandas as pd
import re

def parse_name(name):
    m = {
        'genre': re.search(r'g([A-Z]{2})', name),
        'situation': re.search(r'_s([A-Z]{2})_', name),
        'dancer': re.search(r'_d(\d+)_', name),
        'music': re.search(r'_m([A-Z]{2}\d)', name),
        'choreo': re.search(r'_ch(\d+)', name),
    }
    row = {k: (v.group(1) if v else 'NA') for k, v in m.items()}
    row['tempo_class'] = int(row['music'][-1]) if row['music'] != 'NA' and row['music'][-1].isdigit() else -1
    row['base_seq'] = re.sub(r'(_slice.*|_win.*|_\d+)$', '', name)
    return row

meta = pd.DataFrame([{'idx': i, 'name': n, **parse_name(n)} for i, n in enumerate(test_names)])
parse_ok = (meta['genre'] != 'NA').mean()
print(f'Parse-Rate: {parse_ok:.1%} | Fenster: {len(meta)}')
print(meta['genre'].value_counts().to_dict())
print('Tempoklassen:', sorted(meta['tempo_class'].unique()))

N_STIM = min(512, len(meta))
per_genre = max(1, N_STIM // meta['genre'].nunique())
sel = meta.groupby('genre')['idx'].apply(lambda s: s.sample(min(len(s), per_genre), random_state=SEED))
stim = meta[meta['idx'].isin(sel.tolist())].reset_index(drop=True)
stim.to_csv(DATA_DIR / 'planner_stimulus_suite.csv', index=False)
print('Stimulus-Suite:', len(stim), '→', DATA_DIR / 'planner_stimulus_suite.csv')
stim.head()

## 8) Layer-Stack auswählen

Der Kern ist ein `nn.TransformerEncoder` mit 8 identischen Layern und **`batch_first=True`** — Layer-Outputs sind `[B, 150, 512]`. Wir hooken Anfang / Viertel / Mitte / Dreiviertel / Ende. (Die `enable_nested_tensor`-UserWarning kommt von `norm_first=True` und ist harmlos.)

In [ ]:
LAYERS = CORE.encoder.layers   # nn.ModuleList, 8x TransformerEncoderLayer (batch_first=True)
n_layers = len(LAYERS)
SELECTED_LAYERS = sorted(set([0, n_layers // 4, n_layers // 2, (3 * n_layers) // 4, n_layers - 1]))
print(f'Layer: {n_layers} | ausgewaehlt: {SELECTED_LAYERS} | Layer-Klasse: {type(LAYERS[0]).__name__}')

## 9) Activation-Recorder

Context-Manager mit Forward-Hooks. Musik ist kein kausaler Text — es gibt kein sinnvolles „letztes Token", deshalb **Mean-Pooling über die 150 Frames** (optional `pool='none'` für die volle `[150, d]`-Karte, z. B. für spätere Beat-Analysen).

In [ ]:
class ActivationRecorder:
    def __init__(self, layers, layer_indices, pool='mean'):
        self.layers, self.layer_indices, self.pool = layers, list(layer_indices), pool
        self.cache, self.handles = {}, []

    def _extract(self, output):
        h = output[0] if isinstance(output, (tuple, list)) else output
        if h.dim() == 3 and h.shape[0] == SEQ_LEN and h.shape[1] != SEQ_LEN:
            h = h.transpose(0, 1)  # [T,B,D] -> [B,T,D]; greift hier nie (batch_first=True)
        return h

    def _hook(self, idx):
        def fn(module, inputs, output):
            h = self._extract(output).detach().float()
            self.cache[idx] = (h.mean(dim=1) if self.pool == 'mean' else h).cpu()
        return fn

    def __enter__(self):
        self.handles = [self.layers[i].register_forward_hook(self._hook(i)) for i in self.layer_indices]
        return self

    def __exit__(self, *a):
        for h in self.handles: h.remove()

@torch.no_grad()
def capture(music_batch, layer_indices=None, pool='mean', y=None, t=None):
    layer_indices = SELECTED_LAYERS if layer_indices is None else layer_indices
    with ActivationRecorder(LAYERS, layer_indices, pool) as rec:
        core_forward(music_batch, y=y, t=t)
    return {i: rec.cache[i].numpy() for i in layer_indices}

test_acts = capture(torch.tensor(test_music[:2]))
{k: v.shape for k, v in test_acts.items()}

## 10) Kontrast-Analyse

Gruppen-Zentroiden statt einzelner Minimalpaare — robuster: **Genre** (gBR vs gHO), **Tempo** (Klasse 0 vs 5) und **Situation** (sBM vs sFM). Gemessen werden L2-Distanz und Cosine-Ähnlichkeit der mittleren Aktivierung je Layer. *Hinweis: In diesem Test-Split gibt es nur sBM-Fenster — der Situations-Kontrast wird automatisch übersprungen (Guard unten).*

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def group_centroids(df_a, df_b, batch=32):
    def acts_for(df):
        sums = {i: 0 for i in SELECTED_LAYERS}; n = 0
        for s in range(0, len(df), batch):
            mb = torch.tensor(test_music[df['idx'].values[s:s+batch]])
            a = capture(mb)
            for i in SELECTED_LAYERS: sums[i] = sums[i] + a[i].sum(0)
            n += len(a[SELECTED_LAYERS[0]])
        return {i: sums[i] / n for i in SELECTED_LAYERS}
    return acts_for(df_a), acts_for(df_b)

def cosine(a, b, eps=1e-9):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + eps))

CONTRASTS = [
    ('genre', 'gBR', 'gHO', stim[stim.genre == 'BR'], stim[stim.genre == 'HO']),
    ('tempo', 'langsam(0)', 'schnell(5)', stim[stim.tempo_class == 0], stim[stim.tempo_class == 5]),
    ('situation', 'sBM', 'sFM', stim[stim.situation == 'BM'], stim[stim.situation == 'FM']),
]

rows = []
for axis, la, lb, da, db in CONTRASTS:
    if len(da) < 3 or len(db) < 3:
        print(f'{axis}: zu wenige Beispiele ({len(da)}/{len(db)}) — uebersprungen'); continue
    ca, cb = group_centroids(da, db)
    for i in SELECTED_LAYERS:
        d = ca[i] - cb[i]
        rows.append({'axis': axis, 'a': la, 'b': lb, 'layer': i,
                     'l2_diff': float(np.linalg.norm(d)),
                     'cosine': cosine(ca[i], cb[i]),
                     'n_a': len(da), 'n_b': len(db)})

contrast_df = pd.DataFrame(rows)
contrast_df.to_csv(OUT_DIR / 'planner_contrast_layer_metrics.csv', index=False)
display(contrast_df)

plt.figure(figsize=(8, 4))
sns.lineplot(data=contrast_df, x='layer', y='l2_diff', hue='axis', marker='o')
plt.title('Planner: Kontrast-Staerke pro Layer (Gruppen-Zentroiden)')
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUT_DIR / 'planner_contrast_l2_by_layer.png', dpi=160)
plt.show()

## 11) Aktivierungs-Datensatz für Probes (→ Drive)

In [ ]:
from tqdm.auto import tqdm

per_layer = {i: [] for i in SELECTED_LAYERS}
BATCH = 32
for s in tqdm(range(0, len(stim), BATCH)):
    mb = torch.tensor(test_music[stim['idx'].values[s:s+BATCH]])
    a = capture(mb)
    for i in SELECTED_LAYERS:
        per_layer[i].append(a[i])
ACTS = {i: np.concatenate(v).astype('float32') for i, v in per_layer.items()}

np.savez_compressed(ACT_DIR / 'planner_mean_activations.npz',
                    **{f'layer_{k}': v for k, v in ACTS.items()})
stim.to_csv(ACT_DIR / 'planner_activation_metadata.csv', index=False)
print({k: v.shape for k, v in ACTS.items()})
print('Gespeichert →', ACT_DIR)

## 12) Lineare Probes (Genre, Tempoklasse) + Permutations-Kontrolle

**v2-Fix:** Gruppiert wird nach **Musik-ID** (`music`-Spalte), nicht mehr nach `base_seq`. Der Modellinput ist bei `t = max` und fixem Noise-`y` **nur die Musik**, und derselbe Song (z. B. `mBR0`) kommt bei mehreren Tänzern/Choreos vor — `base_seq`-Gruppierung ließe identische Musikfeatures gleichzeitig in Train und Test (Leakage). Mit 6 Songs pro Genre wird der Split gröber (~1–2 ausgehaltene Songs pro Genre): verrauschter, aber ehrlich — und beantwortet die interessantere Frage, ob das Modell *Genre* kodiert oder nur *Song-Identität*.

**v2.1-Fix:** Der Test-Split enthält nur sBM-Fenster — die Situations-Probe hat damit nur eine Klasse und wird automatisch übersprungen (daran crashte der erste Lauf: `ValueError: … only one class: 'BM'`). Als **Negativkontrolle** dient stattdessen die Genre-Probe mit **zufällig permutierten Labels**: erwartet ist Chance-Niveau; deutlich mehr ⇒ Artefakt in der Auswertungs-Pipeline. Im Plot: gestrichelt rot = 1/K-Chance, gepunktet grau = Majority-Baseline im Test-Split.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

def run_probe(label_col, y_override=None, tag=None):
    tag = tag or label_col
    mask = stim[label_col].astype(str) != '-1'
    y = (stim.loc[mask, label_col].astype(str).values if y_override is None
         else np.asarray(y_override)[mask.values])
    # v2.1-Guard: Labels mit nur einer Klasse (z. B. situation: Test-Split ist rein sBM) ueberspringen.
    if len(np.unique(y)) < 2:
        print(f'{tag}: nur eine Klasse ({np.unique(y)[0]}) — uebersprungen')
        return None
    # v2: Gruppierung nach Musik-ID — derselbe Song darf nie in Train UND Test liegen.
    groups = stim.loc[mask, 'music'].values
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
                  .split(np.zeros(len(y)), y, groups))
    if len(np.unique(y[tr])) < 2 or len(np.unique(y[te])) < 2:
        print(f'{tag}: degenerierter Group-Split (Train/Test einklassig) — uebersprungen')
        return None
    _, counts = np.unique(y[te], return_counts=True)
    majority = counts.max() / counts.sum()
    recs = []
    for i in SELECTED_LAYERS:
        X = ACTS[i][mask.values]
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight='balanced'))
        clf.fit(X[tr], y[tr])
        recs.append({'label': tag, 'layer': i,
                     'accuracy': accuracy_score(y[te], clf.predict(X[te])),
                     'chance': 1.0 / len(np.unique(y)),
                     'majority': majority,
                     'n_test': len(te)})
    return pd.DataFrame(recs)

# Negativkontrolle: Genre-Labels permutieren — bricht die X-y-Kopplung, erwartet: Chance-Niveau.
genre_shuffled = np.random.RandomState(SEED).permutation(stim['genre'].astype(str).values)

probes = [run_probe('genre'), run_probe('tempo_class'), run_probe('situation'),
          run_probe('genre', y_override=genre_shuffled, tag='genre_shuffled')]
probe_df = pd.concat([p for p in probes if p is not None], ignore_index=True)
probe_df.to_csv(OUT_DIR / 'planner_linear_probes.csv', index=False)
display(probe_df)

In [ ]:
g = sns.catplot(data=probe_df, x='layer', y='accuracy', col='label', kind='bar',
                color='#4c78a8', height=3.4, aspect=1.0)
for ax in g.axes.flat:
    lbl = ax.get_title().split(' = ')[-1]
    sub = probe_df[probe_df['label'] == lbl]
    ax.axhline(sub['chance'].iloc[0], ls='--', c='red', lw=1)
    ax.axhline(sub['majority'].iloc[0], ls=':', c='gray', lw=1)
    ax.set_ylim(0, 1)
g.figure.suptitle('Lineare Probes im Residual Stream des Planners', y=1.04)
g.figure.savefig(OUT_DIR / 'planner_linear_probes.png', dpi=160, bbox_inches='tight')
plt.show()

## 13) D3PM-Spezialität: Wann legen sich Entscheidungen fest?

Ein Pre-Hook auf dem Denoiser-Kern protokolliert bei jedem Reverse-Schritt den Label-Zustand `y_t` — der Sampler ruft den Kern genau **1× pro Schritt** mit `y_t` als `[B, 150]`-Long-Tensor auf (verifiziert in `UniformD3PM.sample`). Gemessen wird pro Schritt der Anteil der Frames, die schon dem finalen Plan entsprechen — getrennt für **Transitions** (Label 0) und **Movements** (1–100). Legt sich zuerst die Struktur fest oder zuerst der Inhalt?

**Interpretations-Hinweis:** `sample()` zieht in jedem Schritt *alle* Positionen neu aus `Categorical(logits)` — nichts ist je hart fixiert; die Kurve misst, wie früh die Logits scharf werden (effektive Kristallisation). Für reproduzierbare Rollouts: `deterministic=True`. **v2:** Mittel über mehrere Sequenzen (graue Linien = Einzelsequenzen).

In [ ]:
print('Sampler:', inspect.signature(PLANNER.sample))

traj_states = []

def pre_hook(module, args, kwargs):
    for v in list(args) + list(kwargs.values()):
        if torch.is_tensor(v) and v.dtype == torch.long and v.dim() == 2 and v.shape[-1] == SEQ_LEN:
            traj_states.append(v[0].detach().cpu().clone())
            break

def record_trajectory(idx, deterministic=False):
    traj_states.clear()
    music_1 = torch.tensor(test_music[idx:idx+1]).to(DEVICE).float()
    h = CORE.register_forward_pre_hook(pre_hook, with_kwargs=True)
    try:
        with torch.no_grad():
            plan = PLANNER.sample(music_1, deterministic=deterministic)
    finally:
        h.remove()
    final = plan[0].detach().cpu()
    rows = []
    for step, y in enumerate(traj_states):
        match = (y == final)
        rows.append({'idx': idx, 'step': step,
                     'match_all': match.float().mean().item(),
                     'match_transitions': match[final == 0].float().mean().item() if (final == 0).any() else np.nan,
                     'match_movements': match[final > 0].float().mean().item() if (final > 0).any() else np.nan})
    return pd.DataFrame(rows), final

In [ ]:
N_TRAJ = 16   # v2: ueber mehrere Sequenzen mitteln statt nur einer
trajs = []
for idx in stim['idx'].head(N_TRAJ):
    df_i, final = record_trajectory(int(idx))
    trajs.append(df_i)
traj_df = pd.concat(trajs, ignore_index=True)
traj_df.to_csv(OUT_DIR / 'planner_denoising_trajectory.csv', index=False)
print(f'{traj_df["idx"].nunique()} Sequenzen | {traj_df["step"].max() + 1} Reverse-Schritte')

mean_df = traj_df.groupby('step').mean(numeric_only=True).reset_index()
plt.figure(figsize=(8, 4))
for _, sub in traj_df.groupby('idx'):
    plt.plot(sub['step'], sub['match_all'], color='gray', alpha=0.15, lw=0.7)
for col, lbl in [('match_all', 'alle Frames'), ('match_transitions', 'Transitions (Label 0)'),
                 ('match_movements', 'Movements (1-100)')]:
    plt.plot(mean_df['step'], mean_df[col], label=lbl, lw=2)
plt.xlabel('Reverse-Diffusionsschritt'); plt.ylabel('Anteil = finaler Plan')
plt.title(f'Entscheidungs-Kristallisation im D3PM (Mittel ueber {N_TRAJ} Sequenzen)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUT_DIR / 'planner_denoising_trajectory.png', dpi=160)
plt.show()

## 14) Kausal-Skelett: Music-Swap & Aktivierungs-Patching

(a) **Music-Swap** als Verhaltensbaseline: Plan mit Musik A vs. Musik B (anderes Genre) — wie stark ändern sich Transition-Anteil und Movement-Verteilung? (b) **Aktivierungs-Patch** (Skelett, nach den Probes mit dem besten Layer füllen): Layer-Outputs sind `[B, 150, 512]` (`batch_first=True`), `dim=1` ist also die **Zeitachse** — der Mean-Shift im Skelett ist dimensionsrichtig.

In [ ]:
def plan_stats(music_tensor, deterministic=False):
    with torch.no_grad():
        p = PLANNER.sample(music_tensor.to(DEVICE).float(), deterministic=deterministic)
    p = p[0].detach().cpu()
    segs = int((p[1:] != p[:-1]).sum()) + 1
    return {'transition_frac': float((p == 0).float().mean()),
            'n_segments': segs,
            'top_moves': torch.bincount(p[p > 0], minlength=NUM_CLASSES).topk(3).indices.tolist()}

a_row = stim[stim.genre == 'BR'].iloc[0]
b_row = stim[stim.genre == 'HO'].iloc[0]
m_a = torch.tensor(test_music[int(a_row['idx']):int(a_row['idx'])+1])
m_b = torch.tensor(test_music[int(b_row['idx']):int(b_row['idx'])+1])
print('Musik A (gBR):', plan_stats(m_a))
print('Musik B (gHO):', plan_stats(m_b))

In [ ]:
# --- Aktivierungs-Patch-Skelett (nach den Probes mit dem besten Layer fuellen) ---
# Layer-Outputs sind [B, 150, 512] (batch_first=True) -> dim=1 ist die Zeitachse.
# BEST_LAYER = 4
# source_centroid = torch.tensor(ACTS[BEST_LAYER][(stim.genre == 'BR').values].mean(0))
# def patch_hook(module, inputs, output):
#     h = output[0] if isinstance(output, (tuple, list)) else output
#     h = h + (source_centroid.to(h.device, h.dtype) - h.mean(dim=1, keepdim=True))
#     return (h,) + tuple(output[1:]) if isinstance(output, tuple) else h
# handle = LAYERS[BEST_LAYER].register_forward_hook(patch_hook)
# try:
#     print('gepatcht:', plan_stats(m_b))
# finally:
#     handle.remove()

## 15) Rhythmus-Zeit-Probes (v3)

Antwort auf den 0.0-Befund aus Abschnitt 12. Der war ein **Struktur-Artefakt**: In diesem Test-Split hat jedes Genre genau einen Song, Musik-ID ≡ Genre — der Group-Split hält also ganze Genres aus dem Training, und eine im Training fehlende Klasse ist für die Probe *prinzipiell* unvorhersagbar (deshalb exakt 0.0 statt Chance). Keine Aussage über den Residual Stream.

Der Ausweg (JoruriPuppet-Prinzip „Beat-Tabelle"): **zeitvariable Targets**, die innerhalb jedes Songs variieren — der strenge Musik-ID-Split bleibt, Aliasing ist unmöglich. Targets aus der Beat-Spur der Musikfeatures (Layout verifiziert in `data/audio_extraction/baseline_features.py`: `[Envelope | 20×MFCC | 12×Chroma | Peak | Beat]` → Beat = Dim 34):

- **beat** (binär je Frame) — *Positiv-Kontrolle*: steht direkt im Input, muss überall lesbar sein.
- **phase** (Position zwischen zwei Beats, als sin/cos-Regression) — steht **nicht** im Input; der Input hat nur Impulse an Beat-Frames. Phase an einem Nicht-Beat-Frame erfordert zeitliche Integration durch Attention.
- **ibi** (lokales Tempo = Inter-Beat-Intervall in Frames) und **to_next_beat** — ebenfalls Integrationstargets.

Baselines, die die Aktivierungs-Probes schlagen müssen: rohe Musikframes (35 dim), ±8-Frame-Kontextfenster (595 dim; kann Phase *lokal* teilweise ableiten), Positions-One-Hot (150 dim; fängt „Phase-aus-Fensterposition-memorieren" ab). Erst *Aktivierung ≫ alle Baselines* belegt Integration durch das **Modell** statt Durchleitung.

In [ ]:
FPS = 60   # AIST++-Features laufen mit 60 fps; die Targets unten sind fps-agnostisch (Frames)
ENV_DIM, PEAK_DIM, BEAT_DIM = 0, 33, 34   # Layout: [env | mfcc20 | chroma12 | peak | beat]

bd = test_music[:, :, BEAT_DIM]
thr = 0.5 * (bd.max() + bd.min())   # robust, falls die Spur skaliert sein sollte
beat_grid = bd > thr
if len(np.unique(bd)) > 2:
    print(f'Hinweis: Beat-Spur nicht binaer ({len(np.unique(bd))} Werte) — Schwellwert = {thr:.3f}')
assert beat_grid.any() and not beat_grid.all(), 'Beat-Spur (Dim 34) unplausibel — Feature-Layout pruefen!'

n_beats_win = beat_grid[stim['idx'].values].sum(1)
print(f'Beats pro Fenster: min {int(n_beats_win.min())} / median {int(np.median(n_beats_win))} / max {int(n_beats_win.max())}')

def rhythm_targets(beats):
    T = len(beats); idx = np.flatnonzero(beats)
    phase = np.full(T, np.nan); ibi = np.full(T, np.nan); to_next = np.full(T, np.nan)
    for a, b in zip(idx[:-1], idx[1:]):
        f = np.arange(a, b)
        phase[f] = (f - a) / (b - a); ibi[f] = b - a; to_next[f] = b - f
    return phase, ibi, to_next

tt = [rhythm_targets(b) for b in beat_grid[stim['idx'].values]]
PHASE = np.stack([t[0] for t in tt]); IBI = np.stack([t[1] for t in tt]); TONEXT = np.stack([t[2] for t in tt])
VALID = ~np.isnan(PHASE)
print(f'Valide Frames: {VALID.mean():.1%} | medianes IBI: {np.nanmedian(IBI):.0f} Frames '
      f'(~{60 * FPS / np.nanmedian(IBI):.0f} BPM bei {FPS} fps)')

w0 = 0
fig, ax = plt.subplots(figsize=(9, 2.6))
env = test_music[stim['idx'].iloc[w0], :, ENV_DIM]
ax.plot(env / (np.abs(env).max() + 1e-9), c='gray', alpha=0.6, label='Envelope (norm.)')
ax.vlines(np.flatnonzero(beat_grid[stim['idx'].iloc[w0]]), 0, 1, color='red', lw=1, label='Beat')
ax.plot(PHASE[w0], c='#4c78a8', lw=1.2, label='Phase-Target')
ax.set_xlabel('Frame'); ax.set_title(f'Rhythmus-Targets: {stim["name"].iloc[w0]}')
ax.legend(loc='upper right', fontsize=7); plt.tight_layout(); plt.show()

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import roc_auc_score, r2_score

# Frame-Aktivierungen [Fenster, 150, 512] pro Layer (Musik-only-Zustand wie in Abschnitt 6)
FRAME_LAYERS = SELECTED_LAYERS
frame_acts = {i: [] for i in FRAME_LAYERS}
for s in tqdm(range(0, len(stim), BATCH)):
    mb = torch.tensor(test_music[stim['idx'].values[s:s+BATCH]])
    a = capture(mb, pool='none')
    for i in FRAME_LAYERS:
        frame_acts[i].append(a[i].astype('float32'))
FACTS = {i: np.concatenate(v) for i, v in frame_acts.items()}
print('Frame-Aktivierungen:', {i: v.shape for i, v in FACTS.items()})

RAW = test_music[stim['idx'].values].astype('float32')
K = 8
pad = np.pad(RAW, ((0, 0), (K, K), (0, 0)))
WIN = np.concatenate([pad[:, k:k+SEQ_LEN] for k in range(2 * K + 1)], axis=-1)
POS = np.broadcast_to(np.eye(SEQ_LEN, dtype='float32'), (len(stim), SEQ_LEN, SEQ_LEN)).copy()

groups_w = stim['music'].values
tr_w, te_w = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
                  .split(np.zeros(len(stim)), groups=groups_w))
print(f'Split: {len(tr_w)} Train- / {len(te_w)} Test-Fenster | Test-Songs: {sorted(set(groups_w[te_w]))}')

def flat_masked(X3, Y2, V2, rows):
    X = X3[rows].reshape(-1, X3.shape[-1])
    y = Y2[rows].ravel(); v = V2[rows].ravel()
    return X[v], y[v]

def ridge_score(X3, Y2, V2, metric):
    Xtr, ytr = flat_masked(X3, Y2, V2, tr_w)
    Xte, yte = flat_masked(X3, Y2, V2, te_w)
    sc = StandardScaler().fit(Xtr)
    m = Ridge(alpha=10.0).fit(sc.transform(Xtr), ytr)
    pred = m.predict(sc.transform(Xte))
    return float(roc_auc_score(yte, pred)) if metric == 'auc' else float(r2_score(yte, pred))

BEAT_Y = beat_grid[stim['idx'].values].astype('float32')
ALL_V = np.ones_like(VALID)
FEATURE_SETS = {f'layer_{i}': FACTS[i] for i in FRAME_LAYERS}
FEATURE_SETS.update({'input_raw35': RAW, 'input_window17x35': WIN, 'pos_onehot': POS})

rows_out = []
for fname, X3 in FEATURE_SETS.items():
    rows_out.append({'features': fname, 'target': 'beat', 'metric': 'auc',
                     'score': ridge_score(X3, BEAT_Y, ALL_V, 'auc')})
    r2s = ridge_score(X3, np.sin(2 * np.pi * PHASE), VALID, 'r2')
    r2c = ridge_score(X3, np.cos(2 * np.pi * PHASE), VALID, 'r2')
    rows_out.append({'features': fname, 'target': 'phase', 'metric': 'r2', 'score': (r2s + r2c) / 2})
    rows_out.append({'features': fname, 'target': 'ibi', 'metric': 'r2',
                     'score': ridge_score(X3, IBI, VALID, 'r2')})
    rows_out.append({'features': fname, 'target': 'to_next_beat', 'metric': 'r2',
                     'score': ridge_score(X3, TONEXT, VALID, 'r2')})
    print(fname, 'fertig')

frame_probe_df = pd.DataFrame(rows_out)
frame_probe_df.to_csv(OUT_DIR / 'planner_rhythm_frame_probes.csv', index=False)
display(frame_probe_df.pivot(index='features', columns='target', values='score').round(3))

def get(feat, tgt):
    m = frame_probe_df[(frame_probe_df['features'] == feat) & (frame_probe_df['target'] == tgt)]
    return m['score'].iloc[0]

lay = sorted(FRAME_LAYERS)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for tgt, mk in [('phase', 'o'), ('ibi', 's'), ('to_next_beat', '^')]:
    axes[0].plot(lay, [get(f'layer_{i}', tgt) for i in lay], marker=mk, label=tgt)
for base, ls in [('input_raw35', ':'), ('input_window17x35', '--'), ('pos_onehot', '-.')]:
    axes[0].axhline(get(base, 'phase'), ls=ls, c='gray', lw=1, label=f'phase-Baseline: {base}')
axes[0].set_xlabel('Layer'); axes[0].set_ylabel('R² (gehaltene Songs)')
axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3); axes[0].set_title('Rhythmus-Integration pro Layer')
axes[1].plot(lay, [get(f'layer_{i}', 'beat') for i in lay], marker='o', c='#4c78a8', label='Aktivierungen')
for base, ls in [('input_raw35', ':'), ('input_window17x35', '--'), ('pos_onehot', '-.')]:
    axes[1].axhline(get(base, 'beat'), ls=ls, c='gray', lw=1, label=base)
axes[1].axhline(0.5, c='red', lw=1, alpha=0.5, label='Chance')
axes[1].set_xlabel('Layer'); axes[1].set_ylabel('ROC-AUC'); axes[1].set_ylim(0.4, 1.02)
axes[1].legend(fontsize=7); axes[1].grid(alpha=0.3); axes[1].set_title('Beat-Detektion (Positiv-Kontrolle)')
plt.tight_layout(); plt.savefig(OUT_DIR / 'planner_rhythm_probes.png', dpi=160); plt.show()

### 15c) Counterfactual: Beat-Spur verschieben

Peak- und Beat-Dim werden um ein halbes medianes IBI zirkulär gerollt, alle anderen 33 Dims bleiben identisch. Die auf **clean** trainierte Phase-Probe wird auf den CF-Aktivierungen gegen (a) die *verschobenen* und (b) die *originalen* Phase-Targets getestet. Das trennt die Quelle des Signals: folgt die Probe der expliziten Beat-Spur (Dim 33/34) oder den impliziten Rhythmus-Korrelaten in Envelope/MFCC/Chroma?

In [ ]:
SHIFT = int(round(np.nanmedian(IBI) / 2))
print(f'CF-Shift: {SHIFT} Frames (halbes medianes IBI)')

CF_RAW = RAW.copy()
CF_RAW[:, :, [PEAK_DIM, BEAT_DIM]] = np.roll(CF_RAW[:, :, [PEAK_DIM, BEAT_DIM]], SHIFT, axis=1)

cf_beats = np.roll(beat_grid[stim['idx'].values], SHIFT, axis=1)
cf_tt = [rhythm_targets(b) for b in cf_beats]
CF_PHASE = np.stack([t[0] for t in cf_tt])
V_JOINT = VALID & ~np.isnan(CF_PHASE)

cf_facts = {i: [] for i in FRAME_LAYERS}
for s in range(0, len(stim), BATCH):
    a = capture(torch.tensor(CF_RAW[s:s+BATCH]), pool='none')
    for i in FRAME_LAYERS:
        cf_facts[i].append(a[i].astype('float32'))
CF_FACTS = {i: np.concatenate(v) for i, v in cf_facts.items()}

def phase_r2(models, X3, PH, rows):
    sc, msin, mcos = models
    X, ys = flat_masked(X3, np.sin(2 * np.pi * PH), V_JOINT, rows)
    _, yc = flat_masked(X3, np.cos(2 * np.pi * PH), V_JOINT, rows)
    Xs = sc.transform(X)
    return (r2_score(ys, msin.predict(Xs)) + r2_score(yc, mcos.predict(Xs))) / 2

cf_out = []
for i in FRAME_LAYERS:
    Xtr, ys_tr = flat_masked(FACTS[i], np.sin(2 * np.pi * PHASE), V_JOINT, tr_w)
    _, yc_tr = flat_masked(FACTS[i], np.cos(2 * np.pi * PHASE), V_JOINT, tr_w)
    sc = StandardScaler().fit(Xtr)
    models = (sc,
              Ridge(alpha=10.0).fit(sc.transform(Xtr), ys_tr),
              Ridge(alpha=10.0).fit(sc.transform(Xtr), yc_tr))
    cf_out.append({'layer': i,
                   'clean_r2': phase_r2(models, FACTS[i], PHASE, te_w),
                   'cf_vs_shifted_r2': phase_r2(models, CF_FACTS[i], CF_PHASE, te_w),
                   'cf_vs_original_r2': phase_r2(models, CF_FACTS[i], PHASE, te_w)})
cf_df = pd.DataFrame(cf_out)
cf_df.to_csv(OUT_DIR / 'planner_phase_cf_transfer.csv', index=False)
display(cf_df.round(3))
print('Lesart: cf_vs_shifted >> cf_vs_original -> Phase-Signal folgt der expliziten Beat-Spur.')
print('        cf_vs_original >> cf_vs_shifted -> Phase-Signal stammt aus Envelope/MFCC/Chroma.')

### 15d) Verhaltens-Check: Rasten Transitions auf Beats ein?

Segmentgrenzen der gesampelten Pläne (`deterministic=True`) gegen Beat-Frames; Kontrolle über zufällige zirkuläre Beat-Shifts (**q** = Anteil der Zufalls-Shifts mit schlechterem Alignment: q≈1 stark aligniert, q≈0.5 Zufall). Unter CF-Musik zeigt der Vergleich „alte vs. verschobene Beats", ob auch das *Verhalten* der expliziten Beat-Spur folgt.

In [ ]:
def transition_beat_alignment(plan, beats, n_shift=200, rng=None):
    p = np.asarray(plan)
    bounds = np.flatnonzero(p[1:] != p[:-1]) + 1
    bidx = np.flatnonzero(beats)
    if len(bounds) == 0 or len(bidx) == 0:
        return np.nan, np.nan
    d = np.abs(bounds[:, None] - bidx[None, :]).min(1).mean()
    rng = rng if rng is not None else np.random.RandomState(SEED)
    null = []
    for _ in range(n_shift):
        sb = np.flatnonzero(np.roll(beats, rng.randint(1, len(beats))))
        null.append(np.abs(bounds[:, None] - sb[None, :]).min(1).mean())
    return float(d), float(np.mean(d < np.array(null)))

N_ALIGN = 24
rng = np.random.RandomState(SEED)
align_rows = []
for r in range(min(N_ALIGN, len(stim))):
    with torch.no_grad():
        p_clean = PLANNER.sample(torch.tensor(RAW[r:r+1]).to(DEVICE), deterministic=True)[0].cpu().numpy()
        p_cf = PLANNER.sample(torch.tensor(CF_RAW[r:r+1]).to(DEVICE), deterministic=True)[0].cpu().numpy()
    b_clean = beat_grid[stim['idx'].iloc[r]]
    b_cf = np.roll(b_clean, SHIFT)
    d_cc, q_cc = transition_beat_alignment(p_clean, b_clean, rng=rng)
    d_fo, _ = transition_beat_alignment(p_cf, b_clean, rng=rng)
    d_fs, q_fs = transition_beat_alignment(p_cf, b_cf, rng=rng)
    align_rows.append({'name': stim['name'].iloc[r], 'd_clean': d_cc, 'q_clean': q_cc,
                       'd_cf_vs_alt': d_fo, 'd_cf_vs_verschoben': d_fs, 'q_cf_verschoben': q_fs})
align_df = pd.DataFrame(align_rows)
align_df.to_csv(OUT_DIR / 'planner_beat_alignment.csv', index=False)
print(align_df[['d_clean', 'd_cf_vs_alt', 'd_cf_vs_verschoben']].mean().round(2))
print(f'q (Anteil Zufalls-Shifts schlechter): clean {align_df["q_clean"].mean():.2f} | '
      f'CF vs. verschobene Beats {align_df["q_cf_verschoben"].mean():.2f}')
display(align_df.head(8))

### 15e) Linear-vs-MLP-Gap für die Phase (v3.1)

Der v3-Lauf zeigte: Phase ist **linear** nicht dekodierbar (Layer-R² ≤ 0), während das ±8-Input-Fenster 0.68 erreicht. Offene Frage: fehlt die Phase im Stream *wirklich*, oder ist sie nur **nichtlinear kodiert**? Test nach dem Notebook-02-Prinzip: dieselbe Probe als kleines MLP (1 Hidden-Layer, 128 Units, Subsample für CPU-Laufzeit; ~5–10 min). `mlp_r2(layer) ≫ linear_r2(layer)` ⇒ Phase existiert nichtlinear (Finetuning aufs *Lesbarmachen* unnötig). Beide ≈ ≤ 0 bei `mlp_r2(window) > 0` ⇒ Phase fehlt im Stream tatsächlich.

In [ ]:
from sklearn.neural_network import MLPRegressor

MLP_SETS = ['layer_0', 'layer_4', 'layer_7', 'input_window17x35']
N_SUB = 12000   # Subsample fuer CPU-Laufzeit

def mlp_phase_score(X3):
    Xtr, ys_tr = flat_masked(X3, np.sin(2 * np.pi * PHASE), VALID, tr_w)
    _, yc_tr = flat_masked(X3, np.cos(2 * np.pi * PHASE), VALID, tr_w)
    Xte, ys_te = flat_masked(X3, np.sin(2 * np.pi * PHASE), VALID, te_w)
    _, yc_te = flat_masked(X3, np.cos(2 * np.pi * PHASE), VALID, te_w)
    sub = np.random.RandomState(SEED).permutation(len(Xtr))[:N_SUB]
    sc = StandardScaler().fit(Xtr[sub])
    out = []
    for ytr, yte in [(ys_tr, ys_te), (yc_tr, yc_te)]:
        mlp = MLPRegressor(hidden_layer_sizes=(128,), max_iter=150, early_stopping=True,
                           n_iter_no_change=5, random_state=SEED)
        mlp.fit(sc.transform(Xtr[sub]), ytr[sub])
        out.append(r2_score(yte, mlp.predict(sc.transform(Xte))))
    return float(np.mean(out))

gap_rows = []
for fname in MLP_SETS:
    gap_rows.append({'features': fname,
                     'linear_r2': get(fname, 'phase'),
                     'mlp_r2': mlp_phase_score(FEATURE_SETS[fname])})
    print(fname, 'fertig')
gap_df = pd.DataFrame(gap_rows)
gap_df.to_csv(OUT_DIR / 'planner_phase_linear_vs_mlp.csv', index=False)
display(gap_df.round(3))
print('Lesart: mlp_r2(layer) >> linear_r2(layer)  -> Phase existiert, aber nichtlinear kodiert.')
print('        beide <= 0 bei mlp_r2(window) > 0  -> Phase fehlt im Stream wirklich.')

## 16) Latente Uhr: Settledness bei festem Diffusionsschritt (v3)

Analogon zu den τ_dlm-Probes aus dem Subliminal-Clocks-Setup — mit einem Twist: Der Diffusionsschritt t ist beim Planner **expliziter Input** (Time-Embedding), eine t-Probe wäre zirkulär. Nicht-trivial ist die frameweise **Settledness**: „Ist dieses Frame schon final?" variiert *bei festem t* von Sample zu Sample und Frame zu Frame — das Time-Embedding ist je Schritt konstant, jede dekodierbare Settledness muss aus dem y_t-Inhalt kommen. AUC deutlich über 0.5 (bei Basisrate ≠ 0/1) heißt: Der Stream führt eine frameweise **Verifikations-Spur**. (Die Probe darf dabei das aktuelle Label-Embedding lesen — genau das ist die Frage: ist *Stabilität* linear markiert?)

In [ ]:
N_TAU = 48
TAU_LAYER = SELECTED_LAYERS[len(SELECTED_LAYERS) // 2]
TAU_STRIDE = 10

tau_stim = stim.sample(N_TAU, random_state=SEED).reset_index(drop=True)

def sample_with_acts(idx):
    states, acts, call = [], [], {'i': 0}
    def y_pre(module, args, kwargs):
        for v in list(args) + list(kwargs.values()):
            if torch.is_tensor(v) and v.dtype == torch.long and v.dim() == 2 and v.shape[-1] == SEQ_LEN:
                if call['i'] % TAU_STRIDE == 0:
                    states.append(v[0].detach().cpu().clone())
                break
    def a_fwd(module, inputs, output):
        h = output[0] if isinstance(output, (tuple, list)) else output
        if call['i'] % TAU_STRIDE == 0:
            acts.append(h[0].detach().float().cpu().numpy())
        call['i'] += 1
    h1 = CORE.register_forward_pre_hook(y_pre, with_kwargs=True)
    h2 = LAYERS[TAU_LAYER].register_forward_hook(a_fwd)
    try:
        with torch.no_grad():
            plan = PLANNER.sample(torch.tensor(test_music[idx:idx+1]).to(DEVICE).float())
    finally:
        h1.remove(); h2.remove()
    steps = list(range(0, call['i'], TAU_STRIDE))
    return steps, states, acts, plan[0].detach().cpu()

per_step, fins = {}, {}
for r in tqdm(range(N_TAU)):
    steps, states, acts, fin = sample_with_acts(int(tau_stim['idx'].iloc[r]))
    fins[r] = fin.numpy()
    for si, y, h in zip(steps, states, acts):
        per_step.setdefault(si, []).append((h, y.numpy(), (y == fin).numpy(), r))

mus_tau = tau_stim['music'].values
tr_i, te_i = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED)
                  .split(np.zeros(N_TAU), groups=mus_tau))

tau_rows = []
for si in sorted(per_step):
    H = np.stack([p[0] for p in per_step[si]])
    S = np.stack([p[2] for p in per_step[si]])
    R = np.array([p[3] for p in per_step[si]])
    tr_m = np.isin(R, tr_i); te_m = np.isin(R, te_i)
    Xtr, ytr = H[tr_m].reshape(-1, H.shape[-1]), S[tr_m].ravel()
    Xte, yte = H[te_m].reshape(-1, H.shape[-1]), S[te_m].ravel()
    row = {'step': si, 't': T_MAX - 1 - si, 'settled_rate': float(S.mean()), 'auc': np.nan}
    if min(ytr.sum(), len(ytr) - ytr.sum()) >= 50 and min(yte.sum(), len(yte) - yte.sum()) >= 50:
        sc = StandardScaler().fit(Xtr)
        m = Ridge(alpha=10.0).fit(sc.transform(Xtr), ytr.astype('float32'))
        row['auc'] = float(roc_auc_score(yte, m.predict(sc.transform(Xte))))
    tau_rows.append(row)
tau_df = pd.DataFrame(tau_rows)
tau_df.to_csv(OUT_DIR / 'planner_settledness_probe.csv', index=False)
display(tau_df.round(3))

plt.figure(figsize=(7, 3.5))
plt.plot(tau_df['step'], tau_df['auc'], marker='o', label=f'Settled-Probe AUC (Layer {TAU_LAYER})')
plt.plot(tau_df['step'], tau_df['settled_rate'], marker='.', ls='--', c='gray', label='Settled-Anteil (Basisrate)')
plt.axhline(0.5, c='red', lw=1, alpha=0.5)
plt.xlabel('Reverse-Schritt (t = 99 − Schritt)'); plt.ylim(0, 1.02)
plt.title('Latente Uhr: Ist „Frame schon final?" linear lesbar?')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUT_DIR / 'planner_settledness_probe.png', dpi=160)
plt.show()

### 16b) Kontrolle: Verifikations-Spur oder nur Label-Identität? (v3.1)

Der v3-Lauf zeigte AUC 0.90–0.99 — aber die gesampelten Pläne sind **transition-lastig** (Abschnitt 14: `transition_frac` 0.8–1.0). Wenn der finale Plan überwiegend Label 0 ist, gilt „settled" ≈ „aktuelles Label ist 0", und das kann eine Probe *trivial* aus dem Label-Embedding lesen — ganz ohne echtes Stabilitäts-Signal. Zwei Kontrollen:

1. **Label-One-Hot-Baseline:** dieselbe Settled-Probe, aber die Features sind nur One-Hot(aktuelles Label y_t) (101-dim). Alles, was diese Baseline erreicht, ist Label-Identität, keine Verifikation.
2. **Movement-only-Auswertung:** AUC nur auf Frames mit finalem Label > 0 — dort hilft „Label ist 0" nicht mehr.

**Echt** ist die Verifikations-Spur nur, wenn `auc_act` die Label-Baseline in *beiden* Auswertungen deutlich schlägt.

In [ ]:
eye = np.eye(NUM_CLASSES, dtype='float32')

def auc_for(X4, S, tr_m, te_m, eval_mask):
    Xtr = X4[tr_m].reshape(-1, X4.shape[-1]); ytr = S[tr_m].ravel()
    Xte = X4[te_m].reshape(-1, X4.shape[-1]); yte = S[te_m].ravel()
    em = eval_mask[te_m].ravel()
    Xte, yte = Xte[em], yte[em]
    if min(ytr.sum(), len(ytr) - ytr.sum()) < 50 or min(yte.sum(), len(yte) - yte.sum()) < 50:
        return np.nan
    sc = StandardScaler().fit(Xtr)
    m = Ridge(alpha=10.0).fit(sc.transform(Xtr), ytr.astype('float32'))
    return float(roc_auc_score(yte, m.predict(sc.transform(Xte))))

rows16b = []
for si in sorted(per_step):
    H = np.stack([p[0] for p in per_step[si]])
    Y = np.stack([p[1] for p in per_step[si]])
    S = np.stack([p[2] for p in per_step[si]])
    R = np.array([p[3] for p in per_step[si]])
    F = np.stack([fins[r] for r in R])
    tr_m = np.isin(R, tr_i); te_m = np.isin(R, te_i)
    X_lab = eye[Y]
    all_mask = np.ones_like(S, dtype=bool)
    mov_mask = F > 0
    rows16b.append({'step': si, 't': T_MAX - 1 - si,
                    'auc_act': auc_for(H, S, tr_m, te_m, all_mask),
                    'auc_label': auc_for(X_lab, S, tr_m, te_m, all_mask),
                    'auc_act_mov': auc_for(H, S, tr_m, te_m, mov_mask),
                    'auc_label_mov': auc_for(X_lab, S, tr_m, te_m, mov_mask),
                    'settled_rate': float(S.mean()),
                    'movement_frac': float(mov_mask.mean())})
ctrl_df = pd.DataFrame(rows16b)
ctrl_df.to_csv(OUT_DIR / 'planner_settledness_controls.csv', index=False)
display(ctrl_df.round(3))

plt.figure(figsize=(8, 4))
plt.plot(ctrl_df['step'], ctrl_df['auc_act'], marker='o', label='Aktivierungen (alle Frames)')
plt.plot(ctrl_df['step'], ctrl_df['auc_label'], marker='s', ls='--', label='Label-One-Hot (alle Frames)')
plt.plot(ctrl_df['step'], ctrl_df['auc_act_mov'], marker='o', alpha=0.6, label='Aktivierungen (nur Movements)')
plt.plot(ctrl_df['step'], ctrl_df['auc_label_mov'], marker='s', ls='--', alpha=0.6, label='Label-One-Hot (nur Movements)')
plt.axhline(0.5, c='red', lw=1, alpha=0.5)
plt.xlabel('Reverse-Schritt'); plt.ylabel('ROC-AUC'); plt.ylim(0.35, 1.02)
plt.title('Settledness: echte Verifikations-Spur vs. Label-Identitaet')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUT_DIR / 'planner_settledness_controls.png', dpi=160)
plt.show()

d_all = (ctrl_df['auc_act'] - ctrl_df['auc_label']).mean()
d_mov = (ctrl_df['auc_act_mov'] - ctrl_df['auc_label_mov']).mean()
print(f'mittlerer Vorsprung Aktivierungen vs. Label-Baseline: alle Frames {d_all:+.3f} | nur Movements {d_mov:+.3f}')
print('Lesart: beide Vorspruenge deutlich > 0  -> echte frameweise Verifikations-Spur.')
print('        Vorsprung ~ 0                   -> die "Uhr" war nur Label-Identitaet.')

## 17) Attention-Programme & Replacement: Wer trägt die Verifikations-Spur? (v3.2)

Werkzeug aus `Erikiss/explaining_attention_heads` („Replacing Attention Heads"), übertragen auf den Planner: Für jeden der 8×8 Attention-Heads werden **symbolische Hypothesen-Programme** getestet — Funktionen, die aus den beobachtbaren Inputs (y_t-Labels, Beat-Spur, Position) ein vorhergesagtes Attention-Muster `[150, 150]` bauen. Passung = **Soft-IoU** `sum(min)/sum(max)` (identisch zu `write_data.ipynb` im Repo), dann **Replacement-Experiment**: Head-Attention durch (a) Uniform (Ablation) und (b) das Best-Fit-Programm ersetzen und den Schaden am **Settledness-Gap** messen.

Anwendung auf die Kippe-Frage aus 16b: Eine *echte* Verifikations-Spur braucht einen Mechanismus, der Information **über Frames hinweg** aggregiert — Kandidaten-Programme sind `same_label` (Label-Konsens-Attention: schaue auf Frames mit gleichem aktuellem Label), `transition_frames` und `label_boundaries`. Reine Durchleitungs-Hypothesen sind `self`, `local_pm8`, `uniform`, `first_frames`, `beat_frames`.

**Entscheidungslogik:** Kollabiert die Settled-AUC (Movement-only, gefrorene Probe) bei Uniform-Ablation bestimmter Heads Richtung Label-Baseline, und sind das gerade die Heads mit `same_label`-/Struktur-Bestfit — und rettet Programm-Replacement den Schaden gegenüber Uniform —, dann ist die Uhr ein echter, lokalisierbarer Mechanismus. Zeigt kein Head Wirkung, war die AUC Durchleitung. Eval am „interessanten Bereich" `EVAL_STEP = 50` (settled_rate ≈ 0.55, maximale Varianz). Benötigt die gelaufenen Abschnitte 16/16b.

In [ ]:
import torch.nn.functional as Fn

try:
    torch.backends.mha.set_fastpath_enabled(False)   # Slow-Path erzwingen, damit need_weights greift
except Exception:
    pass

N_HEADS = LAYERS[0].self_attn.num_heads
EVAL_STEP = 50
T_EVAL = T_MAX - 1 - EVAL_STEP
assert EVAL_STEP in per_step, 'Abschnitt 16 muss vorher gelaufen sein (per_step fehlt).'

# Zustaende am Eval-Schritt aus Abschnitt 16 wiederverwenden
Y_S = np.stack([p[1] for p in per_step[EVAL_STEP]])
S_S = np.stack([p[2] for p in per_step[EVAL_STEP]])
R_S = np.array([p[3] for p in per_step[EVAL_STEP]])
F_S = np.stack([fins[r] for r in R_S])
MUS_S = test_music[tau_stim['idx'].values[R_S]].astype('float32')
BEATS_S = beat_grid[tau_stim['idx'].values[R_S]]
print(f'Eval-Schritt {EVAL_STEP} (t={T_EVAL}) | Samples: {len(Y_S)} | settled_rate: {S_S.mean():.3f}')

class AttnTap:
    # Patcht self_attn.forward aller Layer: liefert per-Head-Gewichte und erlaubt
    # per-Head-Replacement der Attention-Matrix (Uniform oder Programm).
    def __init__(self, layers, override=None, collect=False):
        self.layers, self.override, self.collect = layers, override or {}, collect
        self.weights, self._orig = {}, []

    def __enter__(self):
        for li, lay in enumerate(self.layers):
            mha = lay.self_attn
            orig = mha.forward
            self._orig.append((mha, orig))
            def make(li, mha, orig):
                E, H = mha.embed_dim, mha.num_heads
                dh = E // H
                def fwd(query, key, value, **kw):
                    kw['need_weights'] = True
                    kw['average_attn_weights'] = False
                    out, w = orig(query, key, value, **kw)   # w: [B, H, T, T]
                    ov = self.override.get(li)
                    if ov:
                        B, T = query.shape[0], query.shape[1]
                        vw, vb = mha.in_proj_weight[2 * E:], mha.in_proj_bias[2 * E:]
                        V = Fn.linear(value, vw, vb).view(B, T, H, dh).transpose(1, 2)
                        w2 = w.clone()
                        for h, P in ov.items():
                            w2[:, h] = P.to(w.device, w.dtype)
                        out = mha.out_proj(torch.matmul(w2, V).transpose(1, 2).reshape(B, T, E))
                    if self.collect:
                        self.weights[li] = w.detach()
                    return out, w
                return fwd
            mha.forward = make(li, mha, orig)
        return self

    def __exit__(self, *a):
        for mha, orig in self._orig:
            mha.forward = orig

def forward_at_state(Y, M, t_val, override=None, collect=False):
    y = torch.tensor(np.asarray(Y), dtype=torch.long, device=DEVICE)
    m = torch.tensor(np.asarray(M), dtype=torch.float32, device=DEVICE)
    t = torch.full((len(Y),), t_val, dtype=torch.long, device=DEVICE)
    with AttnTap(LAYERS, override=override, collect=collect) as tap:
        with ActivationRecorder(LAYERS, [TAU_LAYER], pool='none') as rec:
            with torch.no_grad():
                CORE(y, m, t)
    return rec.cache[TAU_LAYER].numpy(), tap.weights

# Sanity: Gewichte kommen an und haben per-Head-Form
_, W0 = forward_at_state(Y_S[:2], MUS_S[:2], T_EVAL, collect=True)
assert W0 and tuple(W0[0].shape) == (2, N_HEADS, SEQ_LEN, SEQ_LEN), f'Attention-Capture fehlgeschlagen: {[(k, tuple(v.shape)) for k, v in W0.items()]}'
print('Attention-Capture OK:', {li: tuple(w.shape) for li, w in sorted(W0.items())})

In [ ]:
# --- 17a) Programm-Bibliothek + Soft-IoU-Bestfit (Methode aus write_data.ipynb) ---
def make_row_stochastic(m):
    m = np.clip(m, 0, None)
    s = m.sum(axis=1, keepdims=True)
    s[s == 0] = 1.0
    return m / s

def iou_score(p, q):
    p = np.clip(p.astype(np.float64), 1e-12, 1.0)
    q = np.clip(q.astype(np.float64), 1e-12, 1.0)
    return float(np.minimum(p, q).sum() / np.maximum(p, q).sum())

def _band(T, k):
    i = np.arange(T)
    return (np.abs(i[None, :] - i[:, None]) <= k).astype(float)

PROGRAMS = {
    'self':              lambda y, b: np.eye(len(y)),
    'prev_frame':        lambda y, b: np.eye(len(y), k=-1),
    'next_frame':        lambda y, b: np.eye(len(y), k=1),
    'local_pm8':         lambda y, b: _band(len(y), 8),
    'uniform':           lambda y, b: np.ones((len(y), len(y))),
    'first_frames':      lambda y, b: np.tile((np.arange(len(y)) < 3).astype(float), (len(y), 1)),
    'same_label':        lambda y, b: (y[None, :] == y[:, None]).astype(float),
    'same_label_noself': lambda y, b: np.clip((y[None, :] == y[:, None]).astype(float) - np.eye(len(y)), 0, 1),
    'transition_frames': lambda y, b: np.tile((y == 0).astype(float), (len(y), 1)),
    'label_boundaries':  lambda y, b: np.tile(np.concatenate([[0.0], (np.diff(y) != 0).astype(float)]), (len(y), 1)),
    'beat_frames':       lambda y, b: np.tile(b.astype(float), (len(y), 1)),
}
prog_names = list(PROGRAMS)

N_IOU = 12
iou_sum = np.zeros((n_layers, N_HEADS, len(PROGRAMS)))
for s in tqdm(range(N_IOU)):
    _, W = forward_at_state(Y_S[s:s+1], MUS_S[s:s+1], T_EVAL, collect=True)
    pmats = [make_row_stochastic(PROGRAMS[p](Y_S[s], BEATS_S[s])) for p in prog_names]
    for li in range(n_layers):
        w = W[li][0].float().cpu().numpy()
        for h in range(N_HEADS):
            for pi, pm in enumerate(pmats):
                iou_sum[li, h, pi] += iou_score(w[h], pm)
IOU = iou_sum / N_IOU
best_pi = IOU.argmax(-1)
best_iou = IOU.max(-1)

bestfit_df = pd.DataFrame([{'layer': li, 'head': h,
                            'best_prog': prog_names[int(best_pi[li, h])],
                            'iou': float(best_iou[li, h])}
                           for li in range(n_layers) for h in range(N_HEADS)])
bestfit_df.to_csv(OUT_DIR / 'planner_head_program_bestfit.csv', index=False)
print(bestfit_df['best_prog'].value_counts().to_dict())

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(best_iou, cmap='viridis', aspect='auto')
for li in range(n_layers):
    for h in range(N_HEADS):
        ax.text(h, li, prog_names[int(best_pi[li, h])][:6], ha='center', va='center',
                fontsize=6, color='white')
ax.set_xlabel('Head'); ax.set_ylabel('Layer')
ax.set_title(f'Best-Fit-Programm je Head (Soft-IoU, Schritt {EVAL_STEP})')
plt.colorbar(im, label='IoU'); plt.tight_layout()
plt.savefig(OUT_DIR / 'planner_head_program_bestfit.png', dpi=160)
plt.show()

In [ ]:
# --- 17b) Replacement-Experiment: Schaden am Settledness-Gap (gefrorene Probe) ---
eye17 = np.eye(NUM_CLASSES, dtype='float32')
tr_m17 = np.isin(R_S, tr_i); te_m17 = np.isin(R_S, te_i)
Y_te, M_te, B_te = Y_S[te_m17], MUS_S[te_m17], BEATS_S[te_m17]
yte17 = S_S[te_m17].ravel()
mov17 = (F_S[te_m17] > 0).ravel()

H_tr, _ = forward_at_state(Y_S[tr_m17], MUS_S[tr_m17], T_EVAL)
sc17 = StandardScaler().fit(H_tr.reshape(-1, H_tr.shape[-1]))
probe17 = Ridge(alpha=10.0).fit(sc17.transform(H_tr.reshape(-1, H_tr.shape[-1])),
                                S_S[tr_m17].ravel().astype('float32'))

def eval_auc17(H_eval):
    pred = probe17.predict(sc17.transform(H_eval.reshape(-1, H_eval.shape[-1])))
    return (float(roc_auc_score(yte17, pred)),
            float(roc_auc_score(yte17[mov17], pred[mov17])))

H_clean, _ = forward_at_state(Y_te, M_te, T_EVAL)
auc_all_clean, auc_mov_clean = eval_auc17(H_clean)

# Label-Baseline am selben Schritt (fix, unabhaengig von Interventionen)
Xl_tr = eye17[Y_S[tr_m17]].reshape(-1, NUM_CLASSES)
scl = StandardScaler().fit(Xl_tr)
pl = Ridge(alpha=10.0).fit(scl.transform(Xl_tr), S_S[tr_m17].ravel().astype('float32'))
pred_l = pl.predict(scl.transform(eye17[Y_te].reshape(-1, NUM_CLASSES)))
auc_all_label = float(roc_auc_score(yte17, pred_l))
auc_mov_label = float(roc_auc_score(yte17[mov17], pred_l[mov17]))
print(f'clean: AUC alle {auc_all_clean:.3f} / mov {auc_mov_clean:.3f} | '
      f'Label-Baseline: alle {auc_all_label:.3f} / mov {auc_mov_label:.3f} | '
      f'Gap (mov): {auc_mov_clean - auc_mov_label:+.3f}')

B_TE = int(te_m17.sum())
P_uni = torch.full((B_TE, SEQ_LEN, SEQ_LEN), 1.0 / SEQ_LEN)
rows17 = []
for li in tqdm(range(n_layers)):
    for h in range(N_HEADS):
        Hu, _ = forward_at_state(Y_te, M_te, T_EVAL, override={li: {h: P_uni}})
        a_all_u, a_mov_u = eval_auc17(Hu)
        pname = prog_names[int(best_pi[li, h])]
        P_prog = torch.tensor(np.stack([make_row_stochastic(PROGRAMS[pname](Y_te[s], B_te[s]))
                                        for s in range(B_TE)]), dtype=torch.float32)
        Hp, _ = forward_at_state(Y_te, M_te, T_EVAL, override={li: {h: P_prog}})
        a_all_p, a_mov_p = eval_auc17(Hp)
        rows17.append({'layer': li, 'head': h, 'best_prog': pname, 'iou': float(best_iou[li, h]),
                       'auc_mov_uniform': a_mov_u, 'auc_mov_program': a_mov_p,
                       'd_mov_uniform': a_mov_u - auc_mov_clean,
                       'd_mov_program': a_mov_p - auc_mov_clean,
                       'rescue': a_mov_p - a_mov_u})
repl_df = pd.DataFrame(rows17)
repl_df.to_csv(OUT_DIR / 'planner_head_replacement_gap.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im0 = axes[0].imshow(repl_df.pivot(index='layer', columns='head', values='d_mov_uniform').values,
                     cmap='RdBu', vmin=-0.25, vmax=0.25, aspect='auto')
axes[0].set_title('Δ Settled-AUC (mov) bei Uniform-Ablation'); axes[0].set_xlabel('Head'); axes[0].set_ylabel('Layer')
plt.colorbar(im0, ax=axes[0])
axes[1].scatter(repl_df['iou'], repl_df['rescue'], c='#4c78a8', s=28)
axes[1].axhline(0, c='gray', lw=1)
axes[1].set_xlabel('Best-Fit-IoU'); axes[1].set_ylabel('Programm-Rescue (AUC mov, vs. Uniform)')
axes[1].set_title('Erklaeren die Programme die Head-Funktion?'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT_DIR / 'planner_head_replacement_gap.png', dpi=160)
plt.show()

top = repl_df.sort_values('d_mov_uniform').head(8)
display(top[['layer', 'head', 'best_prog', 'iou', 'd_mov_uniform', 'd_mov_program', 'rescue']].round(3))
carriers = top[top['d_mov_uniform'] < -0.05]
struct_progs = ('same_label', 'same_label_noself', 'transition_frames', 'label_boundaries')
n_struct = int(carriers['best_prog'].isin(struct_progs).sum())
print(f'Heads mit AUC-Schaden < -0.05 bei Ablation: {len(carriers)} | davon Struktur-Programm-Bestfit: {n_struct}')
print('Lesart: Schaden konzentriert auf wenige Heads mit same_label-/Struktur-Bestfit,')
print('        und Programm-Replacement rettet den Schaden (rescue > 0)')
print('        -> Verifikations-Spur = echter, lokalisierbarer Label-Konsens-Mechanismus.')
print('        Kein Head bewegt die AUC merklich -> Durchleitungs-These gestaerkt.')

## 18) SPD: Parameter-Dekomposition der Uhr (v4)

17b ergab: Auf **Head-Granularität** ist die Verifikations-Spur nicht lokalisierbar (max. −0.014 von +0.31 Gap). Das ist entweder echte Redundanz — oder **Superposition**: ein sparsamer Mechanismus, der quer zu Heads/Neuronen liegt und nur auf der falschen Zerlegungsebene „verteilt" aussieht. Genau dafür ist **SPD** (Stochastic Parameter Decomposition, Goodfire AI; arXiv 2506.20790) gebaut: Jede Ziel-Matrix wird in C Rang-1-Komponenten `V·U` zerlegt, eine **Causal-Importance-Funktion** (CI-Transformer) sagt pro Input und Position vorher, welche Komponenten gebraucht werden, und drei Losses erzwingen (a) Faithfulness (Komponenten summieren zur Original-Matrix), (b) stochastische Rekonstruktion (KL auf den Logits unter zufällig maskierten Komponenten) und (c) Importance-Minimality (L_p-Anneal → Sparsität).

Wir nutzen die Referenz-Implementierung aus `Erikiss/param-decomp` (`nano_param_decomp/run.py`) als Bibliothek — ComponentLinear, CI-Transformer und alle Losses werden direkt importiert; nur Datenverteilung (Zustände aus den Abschnitt-16-Trajektorien) und Trainingsschleife sind Planner-spezifisch. **Ziel-Module: die MLP-Matrizen der Layer 0–4** (dort sitzt das Signal laut 17b/16b), `C=80` Komponenten pro Matrix.

**Pilot-Charakter:** Das Paper trainiert 400k Schritte auf 8 GPUs; wir fahren ~4k Schritte auf T4 (~20–25 min) mit verkleinertem CI-Transformer. Die Zerlegung wird also gröber sein — für die Frage „konzentriert vs. flach" reicht das als erster Test, die Hyperparameter (`C_COMP`, `N_STEPS`, `coeff_imp`) sind zum Nachschärfen exponiert. *Hinweis: Nach dieser Zelle sind die Ziel-Linears durch ComponentLinear-Wrapper ersetzt; im „target"-Modus verhalten sie sich identisch zum Original, spätere Re-Runs früherer Zellen bleiben korrekt.*

**v4.1 (nach dem ersten SPD-Lauf):** Die LM-Default-Koeffizienten passen nicht zur Planner-Domäne — die stochastische KL ist hier ~3 Größenordnungen kleiner als bei LMs, wodurch Importance-Minimality die Rekonstruktion erdrückte (L0 kollabierte in 250 Schritten von 40 auf ~1; 18b zeigte `ci-masked ≈ zero-masked` → 18c uninformativ). v4.1 rebalanciert (`coeff_imp` 2e-4→2e-5, `coeff_stoch` 1→25, `p_end` 0.6, 4000 Schritte, Grad-Clip 0.05) und ersetzt das Qualitäts-Gate durch die aussagekräftige Bedingung: Die Komponenten müssen den **MLP-Anteil der Uhr** (clean − zero-masked) mehrheitlich rekonstruieren. Nebenbefund des ersten Laufs, unabhängig von SPD: Ohne die MLPs der Layer 0–4 hält die Uhr noch AUC 0.727 (clean 0.894) — ein erheblicher Teil läuft über Embedding-/Attention-Pfade.

In [ ]:
%pip install -q wandb
import sys
import subprocess
if not os.path.isdir('/content/param_decomp_repo'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/Erikiss/param-decomp.git',
                    '/content/param_decomp_repo'], check=True)
sys.path.insert(0, '/content/param_decomp_repo')

from nano_param_decomp.run import (Config as NanoConfig, ComponentLinear, CITransformer,
                                   install_components, faithfulness_loss,
                                   importance_minimality_loss, anneal_p,
                                   sample_continuous_masks, sample_uniform_k_subset_routing,
                                   set_wrapper_masks, clear_wrapper_masks, kl_logits, cosine_lr)

C_COMP = 80
N_STEPS = 4000
WARMUP_STEPS = 300
B18 = 16
SPD_MODULES = {f'encoder.layers.{l}.linear{k}': C_COMP for l in range(5) for k in (1, 2)}

# v4.1: Koeffizienten fuer die Planner-Domaene rebalanciert. Die LM-Defaults
# (coeff_imp=2e-4, coeff_stoch=1.0) setzen KL~O(1) voraus; beim Planner ist die
# stochastische KL nur ~0.004 -> Importance-Minimality erdrueckte die Rekonstruktion
# (L0 kollabierte im ersten Lauf in 250 Schritten von 40 auf ~1, ci-masked ~ zero-masked).
cfg18 = NanoConfig(
    C_per_module=SPD_MODULES,
    n_steps=N_STEPS, batch_size=B18, seq_len=SEQ_LEN, seed=SEED,
    main_lr=3e-4, faithfulness_warmup_steps=WARMUP_STEPS,
    coeff_faith=1e7, coeff_imp=2e-5, coeff_stoch=25.0,
    p_end=0.6, grad_clip_components=0.05,
    ci_d_model=256, ci_n_blocks=2, ci_n_heads=4, ci_mlp_hidden=512,
)

# Trainingsverteilung: alle geloggten Zustaende (y_t, Musik, t) aus den 16er-Trajektorien
YS_ALL, MIDX_ALL, TS_ALL = [], [], []
for si in sorted(per_step):
    for (h, y, s, r) in per_step[si]:
        YS_ALL.append(y); MIDX_ALL.append(int(tau_stim['idx'].iloc[r])); TS_ALL.append(T_MAX - 1 - si)
YS_ALL = np.stack(YS_ALL); MIDX_ALL = np.array(MIDX_ALL); TS_ALL = np.array(TS_ALL)
print(f'SPD-Trainingszustaende: {len(YS_ALL)} | Module: {len(SPD_MODULES)} x C={C_COMP}')

rng18 = np.random.RandomState(SEED)
def sample_batch(B):
    idx = rng18.randint(0, len(YS_ALL), size=B)
    y = torch.tensor(YS_ALL[idx], dtype=torch.long, device=DEVICE)
    m = torch.tensor(test_music[MIDX_ALL[idx]], dtype=torch.float32, device=DEVICE)
    t = torch.tensor(TS_ALL[idx], dtype=torch.long, device=DEVICE)
    return y, m, t

torch.manual_seed(SEED)
# Idempotent: bei Re-Run der Zelle werden bestehende Wrapper wiederverwendet.
if isinstance(CORE.get_submodule(next(iter(SPD_MODULES))), ComponentLinear):
    wrappers18 = {n: CORE.get_submodule(n) for n in SPD_MODULES}
    print('ComponentLinear-Wrapper bereits installiert — wiederverwendet.')
else:
    wrappers18 = install_components(CORE, SPD_MODULES)
CORE.to(DEVICE)   # WICHTIG: frische V/U-Parameter der Wrapper auf die GPU holen
d_in18 = {n: int(w.W_target.shape[1]) for n, w in wrappers18.items()}
ci_fn18 = CITransformer(d_in18, SPD_MODULES, cfg18).to(DEVICE)
comp_params = [p for w in wrappers18.values() for p in (w.V, w.U)]
print(f'Komponenten-Parameter: {sum(p.numel() for p in comp_params):,} | '
      f'CI-Transformer: {sum(p.numel() for p in ci_fn18.parameters()):,}')

# Faithfulness-Warmup (nur V/U)
wopt = torch.optim.AdamW(comp_params, lr=cfg18.faithfulness_warmup_lr, weight_decay=0.0)
for _ in tqdm(range(WARMUP_STEPS), desc='faith-warmup'):
    wopt.zero_grad(); faithfulness_loss(wrappers18).backward(); wopt.step()

opt18 = torch.optim.AdamW(comp_params + list(ci_fn18.parameters()), lr=cfg18.main_lr, weight_decay=0.0)
log18 = []
for step in tqdm(range(N_STEPS), desc='SPD'):
    for g in opt18.param_groups:
        g['lr'] = cosine_lr(step, N_STEPS, cfg18.main_lr, cfg18.main_lr_final_frac)
    Yb, Mb, Tb = sample_batch(B18)
    clear_wrapper_masks(wrappers18)
    target_logits = CORE(Yb, Mb, Tb)
    acts = {n: w.last_input for n, w in wrappers18.items()}
    ci_lower, ci_upper, _ = ci_fn18(acts)
    loss_faith = faithfulness_loss(wrappers18)
    loss_imp = importance_minimality_loss(ci_upper, anneal_p(step, N_STEPS, cfg18.p_start, cfg18.p_end),
                                          cfg18.imp_eps, cfg18.imp_beta, 1)
    masks, dmasks = sample_continuous_masks(ci_lower)
    routing = sample_uniform_k_subset_routing(list(wrappers18), (B18, SEQ_LEN), torch.device(DEVICE))
    set_wrapper_masks(wrappers18, masks, dmasks, routing)
    try:
        pred = CORE(Yb, Mb, Tb)
    finally:
        clear_wrapper_masks(wrappers18)
    loss_stoch = kl_logits(pred, target_logits)
    total = cfg18.coeff_faith * loss_faith + cfg18.coeff_imp * loss_imp + cfg18.coeff_stoch * loss_stoch
    opt18.zero_grad(); total.backward()
    torch.nn.utils.clip_grad_norm_(comp_params, cfg18.grad_clip_components)
    opt18.step()
    if step % 250 == 0 or step == N_STEPS - 1:
        with torch.no_grad():
            l0 = float(np.mean([(ci > 0).float().sum(-1).mean().item() for ci in ci_lower.values()]))
        log18.append({'step': step, 'faith': float(loss_faith), 'stoch': float(loss_stoch),
                      'imp': float(loss_imp), 'L0_mean': l0})
        print(log18[-1])
pd.DataFrame(log18).to_csv(OUT_DIR / 'planner_spd_training_log.csv', index=False)

### 18b) Qualität der Dekomposition

Drei Checks, bevor die Attribution etwas wert ist: (1) **Faithfulness** — die Komponenten-Summe muss die Original-Matrizen treffen; (2) **KL** unter CI-Maskierung klein gegenüber Zero-Maskierung; (3) die **Settled-Probe** muss unter CI-maskiertem Forward ihre AUC (~0.894 clean) im Wesentlichen behalten — sonst hat die Zerlegung genau den Mechanismus zerstört, den wir untersuchen wollen.

In [ ]:
Yt = torch.tensor(Y_te, dtype=torch.long, device=DEVICE)
Mt = torch.tensor(M_te, dtype=torch.float32, device=DEVICE)
Tt = torch.full((len(Y_te),), T_EVAL, dtype=torch.long, device=DEVICE)

def spd_forward_settled_auc(masks, dmasks):
    set_wrapper_masks(wrappers18, masks, dmasks, None)
    try:
        with ActivationRecorder(LAYERS, [TAU_LAYER], pool='none') as rec:
            with torch.no_grad():
                logits = CORE(Yt, Mt, Tt)
    finally:
        clear_wrapper_masks(wrappers18)
    H = rec.cache[TAU_LAYER].numpy()
    pred = probe17.predict(sc17.transform(H.reshape(-1, H.shape[-1])))
    return float(roc_auc_score(yte17[mov17], pred[mov17])), logits

with torch.no_grad():
    clear_wrapper_masks(wrappers18)
    tgt_logits = CORE(Yt, Mt, Tt)
    acts_e = {n: w.last_input for n, w in wrappers18.items()}
    CI_EVAL, _, _ = ci_fn18(acts_e)

zeros_d = {n: torch.zeros(ci.shape[:-1], device=ci.device) for n, ci in CI_EVAL.items()}
auc_ci, log_ci = spd_forward_settled_auc({n: ci for n, ci in CI_EVAL.items()}, zeros_d)
auc_zero, log_zero = spd_forward_settled_auc({n: torch.zeros_like(ci) for n, ci in CI_EVAL.items()}, zeros_d)

with torch.no_grad():
    kl_ci = float(kl_logits(log_ci, tgt_logits))
    kl_zero = float(kl_logits(log_zero, tgt_logits))
    faith_final = float(faithfulness_loss(wrappers18))
    l0_mod = {n: float((ci > 0).float().sum(-1).mean()) for n, ci in CI_EVAL.items()}

print(f'Faithfulness (MSE/Element): {faith_final:.3e}')
print(f'KL ci-masked: {kl_ci:.4f} | KL zero-masked: {kl_zero:.4f} (Referenzrahmen)')
print(f'Settled-AUC (mov): clean {auc_mov_clean:.3f} | ci-masked {auc_ci:.3f} | zero-masked {auc_zero:.3f}')
print('L0 pro Modul:', {k.split(".")[-2] + "." + k.split(".")[-1]: round(v, 1) for k, v in l0_mod.items()})
rec_frac = (auc_ci - auc_zero) / max(auc_mov_clean - auc_zero, 1e-9)
print(f'Uhr-Rekonstruktion durch die Komponenten: {rec_frac:.1%} des MLP-Anteils (ci vs. zero vs. clean)')
# v4.1: aussagekraeftiges Gate — die Komponenten muessen den MLP-Anteil der Uhr
# mehrheitlich tragen, sonst ist die Dekomposition degeneriert und 18c uninformativ.
assert auc_ci - auc_zero > 0.5 * (auc_mov_clean - auc_zero), (
    'Degenerierte Dekomposition: ci-masked ~ zero-masked — die Komponenten tragen die Uhr nicht. '
    'coeff_imp senken / coeff_stoch erhoehen / N_STEPS erhoehen und neu trainieren.')

### 18c) Komponenten-Attribution: Ist die Uhr ein sparsames Subnetz?

Analog zu 17b, aber in **Komponenten-Raum**: Für jede lebendige Komponente wird ihre CI-Maske auf 0 gesetzt (alle anderen bleiben auf ihren CI-Werten) und der Schaden an der Movement-Settled-AUC gemessen. **Konzentriert** sich der Schaden auf wenige Komponenten (steile sortierte Kurve, hoher Top-10-Anteil), war die Head-Ebene das falsche Raster — die Uhr ist ein sparsames, superponiertes Subnetz. Bleibt die Kurve **flach** wie bei den Heads, ist die Redundanz-These bestätigt. Zusätzlich: mittlere CI der Top-Komponenten auf settled vs. unsettled Movement-Frames — „Uhr-Komponenten" sollten settledness-selektiv feuern.

In [ ]:
S_te = S_S[te_m17]
F_te = F_S[te_m17]
mov2d = F_te > 0

alive = []
for n, ci in CI_EVAL.items():
    mean_ci = ci.mean(dim=(0, 1))
    for c in torch.nonzero(mean_ci > 1e-3).flatten().tolist():
        alive.append((n, c, float(mean_ci[c])))
print(f'Lebendige Komponenten: {len(alive)} von {sum(SPD_MODULES.values())}')

attr_rows = []
for n, c, mci in tqdm(alive):
    masks = {k: ci.clone() for k, ci in CI_EVAL.items()}
    masks[n][..., c] = 0.0
    auc_c, _ = spd_forward_settled_auc(masks, zeros_d)
    ci_np = CI_EVAL[n][..., c].cpu().numpy()
    attr_rows.append({'module': n, 'comp': c, 'mean_ci': mci,
                      'auc_mov': auc_c, 'd_auc': auc_c - auc_ci,
                      'ci_settled_mov': float(ci_np[mov2d & (S_te > 0.5)].mean()) if (mov2d & (S_te > 0.5)).any() else np.nan,
                      'ci_unsettled_mov': float(ci_np[mov2d & (S_te < 0.5)].mean()) if (mov2d & (S_te < 0.5)).any() else np.nan})
spd_df = pd.DataFrame(attr_rows).sort_values('d_auc')
spd_df.to_csv(OUT_DIR / 'planner_spd_component_attribution.csv', index=False)
display(spd_df.head(15).round(4))

damages = -spd_df['d_auc'].clip(upper=0).values
order = np.sort(damages)[::-1]
top10_share = order[:10].sum() / max(order.sum(), 1e-12)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(order, marker='.', lw=1)
axes[0].set_xlabel('Komponente (sortiert)'); axes[0].set_ylabel('AUC-Schaden bei Ablation')
axes[0].set_title(f'Konzentration: Top-10 tragen {top10_share:.0%} des Gesamtschadens')
axes[0].grid(alpha=0.3)
top15 = spd_df.head(15)
axes[1].scatter(spd_df['ci_unsettled_mov'], spd_df['ci_settled_mov'], s=12, alpha=0.4, c='gray')
axes[1].scatter(top15['ci_unsettled_mov'], top15['ci_settled_mov'], s=40, c='#c44e52', label='Top-15 Schaden')
lim = max(axes[1].get_xlim()[1], axes[1].get_ylim()[1])
axes[1].plot([0, lim], [0, lim], c='gray', lw=1, ls=':')
axes[1].set_xlabel('mittlere CI (unsettled Mov.)'); axes[1].set_ylabel('mittlere CI (settled Mov.)')
axes[1].set_title('Settledness-Selektivitaet der Komponenten'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT_DIR / 'planner_spd_attribution.png', dpi=160)
plt.show()

per_layer = (spd_df.assign(layer=spd_df['module'].str.extract(r'layers\.(\d)').astype(int))
                   .groupby('layer')['d_auc'].sum())
print('Schaden-Summe pro Layer:', per_layer.round(4).to_dict())
print(f'Lesart: top10_share hoch (>50%) und Top-Komponenten settledness-selektiv (ueber der Diagonale)')
print('        -> Uhr = sparsames Subnetz in Superposition; Head-Raster war zu grob.')
print('        top10_share niedrig und Kurve flach -> echte Redundanz auch auf Parameter-Ebene.')

## 19) Jacobian-Lens: J-Spaces, helle Punkte & Geometrie-Variation (v5)

Werkzeug aus `Erikiss/jacobian-lens` (Referenz-Implementierung zum Global-Workspace-Paper), übertragen auf den Planner: Die Lens transportiert eine Mid-Layer-Aktivierung linear in die Endschicht-Basis und dekodiert sie mit dem **eigenen Output-Head** des Modells — `lens_l(h) = OutputHead(J_l · h)` mit `J_l = E[∂h_final/∂h_l]`. Der Estimator folgt `jlens/fitting.py`: One-Hot-Kotangens pro Output-Dimension an **allen** Ziel-Frames zugleich (bidirektionaler Encoder → keine Kausal-Beschränkung, keine Attention-Sink-Maske), Rückwärtspass in Dim-Batches, Mittel über Quell-Frames und Zustände.

- **J-Spaces:** die Top-Singulärräume von `J_l` — die Richtungen im Mid-Layer, die die Endschicht überhaupt erreichen. Hier prüfen wir deine These der *einen identifizierbaren orthogonalen Richtung*: Wie steil fällt das Spektrum ab, und wie liegen die Top-Richtungen zur Settled-Probe-Richtung und zur Top-SPD-Komponente (L4.linear2 c63)?
- **Helle Punkte:** Layer×Frame-Zellen, in denen die Lens-Vorhersage schon mit der finalen Modell-Vorhersage übereinstimmt — dort ist die Entscheidung im Mid-Layer bereits „transportfähig" kodiert. Dazu die Verbindung zur Uhr: Korrelieren helle Frames mit Settledness?
- **Geometrie-Variation (19c):** kontrollierte Eingriffe an der hellsten Mid-Schicht — Skalierung (α·h), Rotation im Top-J-Subraum, Translation entlang J-orthogonaler Richtungen, Zufalls-Translation als Kontrolle. Pro Familie messen wir, was **invariant** bleibt (Plan-Inhalt/Argmax-Struktur = „Grammatik") und was mit der Modifikation **kovariiert** (Uhr-Anzeige der Settled-Probe, Logit-Verteilung = geometrie-gekoppelte Logik). Benötigt Abschnitte 16–18.

In [ ]:
# --- 19a) Lens fitten: J_l per Kotangenten-Estimator (jlens/fitting.py-Schema) ---
MID_LAYERS = [1, 2, 3, 4, 5, 6]
TARGET_LAYER = n_layers - 1
DIM_BATCH = 16
N_FIT = 24

rng19 = np.random.RandomState(SEED)
fit_idx = rng19.choice(len(YS_ALL), size=N_FIT, replace=False)

class LensTap:
    # Faengt Layer-Outputs OHNE detach (Autograd-faehig) ab.
    def __init__(self, layers, indices):
        self.layers, self.indices = layers, indices
        self.acts, self.handles = {}, []
    def __enter__(self):
        for i in self.indices:
            def mk(i):
                def fn(m, inp, out):
                    h = out[0] if isinstance(out, (tuple, list)) else out
                    self.acts[i] = h
                return fn
            self.handles.append(self.layers[i].register_forward_hook(mk(i)))
        return self
    def __exit__(self, *a):
        for h in self.handles:
            h.remove()

D = 512
J = {l: torch.zeros(D, D, device=DEVICE) for l in MID_LAYERS}
n_chunks = (D + DIM_BATCH - 1) // DIM_BATCH
for si in tqdm(fit_idx, desc='J-Fit'):
    y1 = torch.tensor(YS_ALL[si:si+1], dtype=torch.long, device=DEVICE).repeat(DIM_BATCH, 1)
    m1 = torch.tensor(test_music[MIDX_ALL[si:si+1]], dtype=torch.float32, device=DEVICE).repeat(DIM_BATCH, 1, 1)
    # Alle Parameter sind seit Abschnitt 18 eingefroren -> ohne grad-faehigen Input
    # entsteht KEIN Autograd-Graph. Der Musik-Input liefert ihn (Gradienten w.r.t.
    # Zwischen-Aktivierungen sind davon unabhaengig exakt).
    m1.requires_grad_(True)
    t1 = torch.full((DIM_BATCH,), int(TS_ALL[si]), dtype=torch.long, device=DEVICE)
    with LensTap(LAYERS, MID_LAYERS + [TARGET_LAYER]) as tap:
        _ = CORE(y1, m1, t1)
        h_fin = tap.acts[TARGET_LAYER]
        sources = [tap.acts[l] for l in MID_LAYERS]
        for c in range(n_chunks):
            G = torch.zeros_like(h_fin)
            for b in range(DIM_BATCH):
                d = c * DIM_BATCH + b
                if d < D:
                    G[b, :, d] = 1.0   # Kotangens an ALLEN Ziel-Frames (bidirektional)
            grads = torch.autograd.grad(h_fin, sources, grad_outputs=G,
                                        retain_graph=(c < n_chunks - 1))
            for l, g in zip(MID_LAYERS, grads):
                for b in range(DIM_BATCH):
                    d = c * DIM_BATCH + b
                    if d < D:
                        J[l][d] += g[b].mean(0)   # Mittel ueber Quell-Frames
J = {l: (Jl / N_FIT).cpu() for l, Jl in J.items()}
torch.save({f'layer_{l}': Jl for l, Jl in J.items()}, OUT_DIR / 'planner_jacobian_lens.pt')
print('J_l gefittet fuer Layer', MID_LAYERS, '| Norm:', {l: round(float(Jl.norm()), 2) for l, Jl in J.items()})

# Spektren: gibt es EINE dominante Richtung in den mittleren Schichten?
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
SVD = {}
for l in MID_LAYERS:
    U_, S_, Vt_ = np.linalg.svd(J[l].numpy())
    SVD[l] = (U_, S_, Vt_)
    axes[0].semilogy(S_[:64], label=f'Layer {l}')
axes[0].set_xlabel('Singulaerwert-Index'); axes[0].set_ylabel('σ (log)')
axes[0].set_title('J-Space-Spektren der Mid-Layer'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

probe_dir = probe17.coef_ / (np.linalg.norm(probe17.coef_) + 1e-9)
rows19 = []
for l in MID_LAYERS:
    Vt_ = SVD[l][2]
    cos_probe = [float(np.abs(Vt_[k] @ probe_dir)) for k in range(4)]
    row = {'layer': l, 'sigma1/sigma2': float(SVD[l][1][0] / SVD[l][1][1]),
           'eff_rank': float((SVD[l][1].sum() ** 2) / (SVD[l][1] ** 2).sum())}
    row.update({f'cos_v{k+1}_probe': c for k, c in enumerate(cos_probe)})
    rows19.append(row)
try:
    u63 = wrappers18['encoder.layers.4.linear2'].U[63].detach().cpu().numpy()
    u63 = u63 / (np.linalg.norm(u63) + 1e-9)
    for row, l in zip(rows19, MID_LAYERS):
        row['cos_v1_spdc63'] = float(np.abs(SVD[l][2][0] @ u63))
except Exception as e:
    print('SPD-Richtung nicht verfuegbar:', e)
jspace_df = pd.DataFrame(rows19)
jspace_df.to_csv(OUT_DIR / 'planner_jspace_spectra.csv', index=False)
display(jspace_df.round(3))
axes[1].bar(jspace_df['layer'], jspace_df['eff_rank'], color='#4c78a8')
axes[1].set_xlabel('Layer'); axes[1].set_ylabel('effektiver Rang von J_l')
axes[1].set_title('Wie niederdimensional ist der Transport?'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT_DIR / 'planner_jspace_spectra.png', dpi=160)
plt.show()

In [ ]:
# --- 19b) Helle Punkte: Layer x Frame Lens-Uebereinstimmung + Settledness-Link ---
def out_head(h):
    return CORE.output(h)

with torch.no_grad():
    clear_wrapper_masks(wrappers18)
    with LensTap(LAYERS, MID_LAYERS + [TARGET_LAYER]) as tap:
        logits_fin = CORE(Yt, Mt, Tt)
    pred_fin = logits_fin.argmax(-1)                      # [B,150]
    H_mid = {l: tap.acts[l].detach() for l in MID_LAYERS}

bright = {}
lens_settle_corr = {}
S_flat = S_S[te_m17]
for l in MID_LAYERS:
    Jl = J[l].to(DEVICE)
    lens_logits = out_head(H_mid[l] @ Jl.T)               # [B,150,101]
    agree = (lens_logits.argmax(-1) == pred_fin).float()  # [B,150]
    bright[l] = agree.cpu().numpy()
    a = bright[l].ravel()
    s = S_flat.ravel().astype(float)
    lens_settle_corr[l] = float(np.corrcoef(a, s)[0, 1])

bright_map = np.stack([bright[l].mean(0) for l in MID_LAYERS])   # [n_mid, 150]
plt.figure(figsize=(10, 3.5))
plt.imshow(bright_map, aspect='auto', cmap='viridis', vmin=0, vmax=1)
plt.yticks(range(len(MID_LAYERS)), [f'L{l}' for l in MID_LAYERS])
plt.xlabel('Frame'); plt.ylabel('Mid-Layer')
plt.title(f'Helle Punkte: Lens-Uebereinstimmung mit finaler Vorhersage (Schritt {EVAL_STEP})')
plt.colorbar(label='Agreement'); plt.tight_layout()
plt.savefig(OUT_DIR / 'planner_lens_bright_spots.png', dpi=160)
plt.show()

print('mittlere Helligkeit pro Layer:', {l: round(float(bright[l].mean()), 3) for l in MID_LAYERS})
print('Korrelation Helligkeit <-> Settledness:', {l: round(c, 3) for l, c in lens_settle_corr.items()})
L_STAR = max(MID_LAYERS[:4], key=lambda l: bright[l].mean())
print(f'Hellste mittlere Schicht (fuer 19c): L_STAR = {L_STAR}')

### 19c) Geometrie-Variation: invariante Grammatik vs. geometrie-gekoppelte Logik

Vier kontrollierte Eingriffs-Familien an `L_STAR` (Patch des Layer-Outputs im laufenden Forward): **Skalierung** `h → α·h`; **Rotation** im Top-2-J-Subraum um Winkel θ; **Orth-Translation** entlang einer zu den Top-64-J-Zeilenrichtungen orthogonalen Richtung (sollte die Endschicht kaum erreichen — Nullraum-Test); **Zufalls-Translation** gleicher Stärke (Kontrolle). Pro Eingriff drei Messgrößen: **Plan-Agreement** (Argmax-Übereinstimmung mit clean = „Grammatik"), **Uhr-Verschiebung** (mittlere Settled-Probe-Anzeige auf Movement-Frames minus clean) und **Logit-KL**. Invariant unter allen Geometrien = Grammatik; systematisch mit α/θ/β kovariierend = geometrie-gekoppelte Logik (Erwartung nach 18c: die Uhr als Magnituden-Code reagiert auf Skalierung, die Label-Struktur bleibt).

In [ ]:
Jl_star = J[L_STAR].numpy()
U_s, S_s, Vt_s = SVD[L_STAR]
V2 = torch.tensor(Vt_s[:2].T, dtype=torch.float32, device=DEVICE)      # [512,2] Top-2-J-Ebene
row_space = Vt_s[:64]
v_rand = rng19.randn(D); v_rand /= np.linalg.norm(v_rand)
v_orth = rng19.randn(D)
v_orth -= row_space.T @ (row_space @ v_orth)
v_orth /= np.linalg.norm(v_orth)
v_orth_t = torch.tensor(v_orth, dtype=torch.float32, device=DEVICE)
v_rand_t = torch.tensor(v_rand, dtype=torch.float32, device=DEVICE)

def patched_metrics(transform):
    def hook(m, inp, out):
        h = out[0] if isinstance(out, (tuple, list)) else out
        return transform(h)
    hnd = LAYERS[L_STAR].register_forward_hook(hook)
    try:
        with torch.no_grad():
            with ActivationRecorder(LAYERS, [TAU_LAYER], pool='none') as rec:
                logits_p = CORE(Yt, Mt, Tt)
    finally:
        hnd.remove()
    Hp = rec.cache[TAU_LAYER].numpy()
    pred_probe = probe17.predict(sc17.transform(Hp.reshape(-1, D)))
    agree = float((logits_p.argmax(-1) == pred_fin).float().mean())
    clock_shift = float(pred_probe[mov17].mean())
    kl = float(kl_logits(logits_p, logits_fin))
    return agree, clock_shift, kl

base_agree, clock_clean, base_kl = patched_metrics(lambda h: h)

geo_rows = []
for alpha in [0.5, 0.75, 1.25, 1.5, 2.0]:
    a, c, k = patched_metrics(lambda h, al=alpha: al * h)
    geo_rows.append({'familie': 'skalierung', 'param': alpha, 'plan_agree': a,
                     'uhr_shift': c - clock_clean, 'logit_kl': k})
import math as _math
for deg in [15, 30, 60, 90, 180]:
    th = _math.radians(deg)
    R = torch.tensor([[_math.cos(th), -_math.sin(th)], [_math.sin(th), _math.cos(th)]],
                     dtype=torch.float32, device=DEVICE)
    def rot(h, R=R):
        a2 = h @ V2
        return h + (a2 @ (R - torch.eye(2, device=DEVICE)).T) @ V2.T
    a, c, k = patched_metrics(rot)
    geo_rows.append({'familie': 'rotation_topJ', 'param': deg, 'plan_agree': a,
                     'uhr_shift': c - clock_clean, 'logit_kl': k})
for beta in [0.5, 1.0, 2.0]:
    for name, v in [('orth_translation', v_orth_t), ('random_translation', v_rand_t)]:
        def trans(h, b=beta, vv=v):
            scale = h.norm(dim=-1, keepdim=True).mean()
            return h + b * scale * vv
        a, c, k = patched_metrics(trans)
        geo_rows.append({'familie': name, 'param': beta, 'plan_agree': a,
                         'uhr_shift': c - clock_clean, 'logit_kl': k})
geo_df = pd.DataFrame(geo_rows)
geo_df.to_csv(OUT_DIR / 'planner_geometry_variation.csv', index=False)
print(f'clean: plan_agree {base_agree:.3f} | Uhr-Anzeige (mov) {clock_clean:.3f} | KL {base_kl:.5f}')
display(geo_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for fam, g in geo_df.groupby('familie'):
    axes[0].plot(range(len(g)), g['plan_agree'], marker='o', label=fam)
    axes[1].plot(range(len(g)), g['uhr_shift'], marker='o', label=fam)
axes[0].axhline(base_agree, c='gray', lw=1, ls=':')
axes[0].set_ylabel('Plan-Agreement (Grammatik)'); axes[0].set_xlabel('Eingriffs-Staerke (Index)')
axes[1].axhline(0, c='gray', lw=1, ls=':')
axes[1].set_ylabel('Uhr-Verschiebung (Settled-Anzeige, mov)'); axes[1].set_xlabel('Eingriffs-Staerke (Index)')
for ax in axes: ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.suptitle(f'Geometrie-Variation an Layer {L_STAR}')
plt.tight_layout(); plt.savefig(OUT_DIR / 'planner_geometry_variation.png', dpi=160)
plt.show()

inv = geo_df.groupby('familie').agg(agree_min=('plan_agree', 'min'),
                                    uhr_max_abs=('uhr_shift', lambda s: float(np.abs(s).max())),
                                    kl_max=('logit_kl', 'max'))
display(inv.round(4))
print('Lesart: agree_min hoch + uhr_max_abs klein  -> Eingriff trifft nur Geometrie, Inhalt invariant (Grammatik).')
print('        agree_min hoch + uhr_max_abs gross  -> Uhr kovariiert mit Geometrie, Inhalt bleibt -> Uhr = geometrie-gekoppelter Code.')
print('        orth_translation ~ clean            -> Nullraum-Bestaetigung: J-Space traegt, Orthogonales nicht.')

## Ausblick

- **CoAx-artige Head-Ablationen** (Analogon Notebook 04): Damage = Phase-Probe-Fehler + Transition-Beat-Alignment → „Beat-Heads" prinzipiiert identifizieren statt raten.
- **Phase-Steering** (Analogon Notebook 02): Phase-Richtung aus der Probe in Layer L addieren bzw. rollen — verschieben sich die Transitions im gesampelten Plan?
- **Genre-Generalisierung** ist mit diesem Test-Split strukturell unmessbar (1 Song pro Genre). Wer sie will, zieht Probing-Fenster aus `data/atomic_aistpp/train` (mehrere Songs pro Genre) — der Planner bleibt dabei eingefroren.
- **Music-Swap-Matrix** Genre×Genre mit `deterministic=True`; **Noise-Robustheit** der Mean-Probes über mehrere `NOISE_Y`-Seeds.

Alle Artefakte liegen unter `MyDrive/AtomicDance/mechinterp/` und überleben die Session.